In [1]:
import pandas as pd
import numpy as np

<p2>Thiết kế các schema theo mô hình Dimension/Fact Table</p2>

<h2>I. Dimensional Tables</h2>

<p2>Sử dụng các file csv đã được xử lý ở basic-transformation</p2>

In [3]:
#fbref
fbref_matchgoals = pd.read_csv("gs://football_data_etl/football_data_extracted/output-basic/fbref_matchgoals.csv/part-00000-e8501008-4b91-4e70-97fa-f200c5a97ef2-c000.csv")
fbref_matchinfos = pd.read_csv("gs://football_data_etl/football_data_extracted/output-basic/fbref_matchinfos.csv/part-00000-b133a2c9-fee4-4fce-b917-ae96c664aa8e-c000.csv")
#
fbref_matchsquad = pd.read_csv("gs://football_data_etl/football_data_extracted/output-basic/fbref_matchsquad.csv/part-00000-701e4a9b-5a6b-4171-9c82-9857304c7151-c000.csv")
fbref_matchstats = pd.read_csv("gs://football_data_etl/football_data_extracted/output-basic/fbref_matchstats.csv/part-00000-ff109b2d-1c71-4ca4-bc94-12df8a30a86c-c000.csv")
fbref_matchplayerstats = pd.read_csv("gs://football_data_etl/football_data_extracted/output-basic/fbref_matchplayerstats.csv/part-00000-e0143337-4c3d-41e4-9213-a6ffee29f714-c000.csv")

In [6]:
fbref_matchinfos.head(5)

,Match_Id,League,Season,Match_Week,Home_Team,Away_Team,Match_Date,Venue_Day,Venue_Time,Attendance,Stadium,Officials,Link
0,a68e623d,Ligue 1,2018,1,Monaco,Toulouse,2017-08-04,Fri,2024-07-02T20:45:00.000Z,13572.0,Stade Louis II,Clément Turpin,https://fbref.com/en/matches/a68e623d/Monaco-T...
1,37f2c25f,Ligue 1,2018,1,Paris S-G,Amiens,2017-08-05,Sat,2024-07-02T17:15:00.000Z,46898.0,Parc des Princes,Mikael Lesage,https://fbref.com/en/matches/37f2c25f/Paris-Sa...
2,4d28b63b,Ligue 1,2018,1,Lyon,Strasbourg,2017-08-05,Sat,2024-07-02T20:00:00.000Z,1979.0,Groupama Stadium,Ruddy Buquet,https://fbref.com/en/matches/4d28b63b/Lyon-Str...
3,68b9eea2,Ligue 1,2018,1,Saint-Étienne,Nice,2017-08-05,Sat,2024-07-02T20:00:00.000Z,25879.0,Stade Geoffroy-Guichard,François Letexier,https://fbref.com/en/matches/68b9eea2/Saint-Et...
4,7d2eb66d,Ligue 1,2018,1,Metz,Guingamp,2017-08-05,Sat,2024-07-02T20:00:00.000Z,14595.0,Stade Saint-Symphorien,Jérôme Miguelgorry,https://fbref.com/en/matches/7d2eb66d/Metz-Gui...


In [7]:
fbref_matchinfos['Match_Date'] = pd.to_datetime(fbref_matchinfos['Match_Date'])
fbref_matchinfos['Match_Date'] = fbref_matchinfos['Match_Date'].dt.strftime('%Y-%m-%d')
fbref_matchinfos['Venue_Time'] = pd.to_datetime(fbref_matchinfos['Venue_Time'])
fbref_matchinfos['Venue_Time'] = fbref_matchinfos['Venue_Time'].dt.strftime('%Y-%m-%d %H:%M')


In [8]:
#sofifa
sofifa_players_attr = pd.read_csv('gs://football_data_etl/football_data_extracted/output-basic/sofifa_players_attr.csv/part-00000-0474179e-867f-4f31-9d30-40c396f830f5-c000.csv',
                                  on_bad_lines="skip")

<ipython-input-8-8df04426fdee>:2: DtypeWarning: Columns (45) have mixed types. Specify dtype option on import or set low_memory=False.
  sofifa_players_attr = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/fbref/sofifa_players_attr.csv/part-00000-0474179e-867f-4f31-9d30-40c396f830f5-c000.csv',


In [9]:
sofifa_players_attr['Birthday'] = pd.to_datetime(sofifa_players_attr['Birthday'])
sofifa_players_attr['Birthday'] = sofifa_players_attr['Birthday'].dt.strftime('%Y-%m-%d')
sofifa_players_attr['Club_Joined'] = pd.to_datetime(sofifa_players_attr['Club_Joined'])
sofifa_players_attr['Club_Joined'] = sofifa_players_attr['Club_Joined'].dt.strftime('%Y-%m-%d')
sofifa_players_attr['Update_Date'] = pd.to_datetime(sofifa_players_attr['Update_Date'])
sofifa_players_attr['Update_Date'] = sofifa_players_attr['Update_Date'].dt.strftime('%Y-%m-%d')

In [10]:
sofifa_players_attr

,Acceleration,Age,Aggression,Agility,All_Positions,Attacking_Work_Rate,Balance,Ball_Control,Birthday,Body_Type,...,Stamina,Standing_Tackle,Strength,Update_Date,Value,Vision,Volleys,Wage,Weak_Foot,Weight
0,79,27,77,79,RB RWB,High,77,75,1993-04-26,Normal (170-185),...,81.0,77.0,65.0,2020-09-23,€15M,65.0,52.0,€46K,3.0,71.0
1,80,27,48,75,ST LM,High,71,79,1992-09-14,Normal (185+),...,72.0,14.0,76.0,2020-09-23,€15M,65.0,73.0,€65K,4.0,78.0
2,89,24,62,81,LM,High,81,81,1996-03-26,Normal (170-185),...,74.0,37.0,60.0,2020-09-23,€21M,74.0,68.0,€33K,4.0,70.0
3,32,36,83,47,CB,Medium,53,64,1983-12-22,Lean (185+),...,59.0,79.0,82.0,2020-09-23,€2.5M,55.0,34.0,€24K,3.0,81.0
4,30,24,25,32,GK,Medium,29,18,1995-10-31,Normal (185+),...,26.0,10.0,66.0,2020-09-23,€21.5M,42.0,8.0,€20K,2.0,85.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
291042,89,30,63,89,RW,High,91,86,1992-06-15,Unique,...,87.0,43.0,76.0,2023-07-05,€99.5M,84.0,83.0,€260K,3.0,71.0
291043,82,21,87,76,ST,High,74,82,2000-07-21,Unique,...,79.0,53.0,93.0,2023-07-05,€176.5M,74.0,89.0,€240K,3.0,94.0
291044,75,31,75,76,CM CAM,High,78,90,1991-06-28,Unique,...,89.0,66.0,74.0,2023-07-05,€107.5M,94.0,83.0,€350K,5.0,75.0
291045,69,19,51,56,LB,High,58,42,2003-04-04,Normal (170-185),...,59.0,67.0,52.0,2023-06-05,€600K,42.0,31.0,€6K,3.0,73.0


<h3>1. Dim League</h3>

In [11]:
dim_league = pd.DataFrame({'League_Key': [],
                          'League_Name': [],
                          'League_Description': []})

list_league_key = []
list_league_name = fbref_matchinfos['League'].unique().tolist()

for i in range(0,len(list_league_name)):
    list_league_key.append(i+1)

dim_league['League_Key'] = list_league_key
dim_league['League_Name'] = list_league_name

list_league_description = ['Ligue 1,[A] officially known as Ligue 1 Uber Eats for sponsorship reasons,[1][2] is a French professional league for men primary football competition. Administered by the Ligue de Football Professionnel, Ligue 1 is contested by 18 clubs (as of the 2023–24 season) and operates on a system of promotion and relegation from and to Ligue 2.',
                           'The Premier League is the highest level of the English football league system. Contested by 20 clubs, it operates on a system of promotion and relegation',
                           'The Campeonato Nacional de Liga de Primera División, commonly known as the Primera División or LaLiga and officially as LaLiga EA Sports since 2023',
                           'The Bundesliga sometimes referred to as the Fußball-Bundesliga or 1. Bundesliga is a professional association football league in Germany.',
                           'The Serie A officially known as Serie A TIM for sponsorship reasons, is a professional league competition for football clubs located at the top of the']
dim_league['League_Description'] = list_league_description


In [12]:
dim_league

,League_Key,League_Name,League_Description
0,1,Ligue 1,"Ligue 1,[A] officially known as Ligue 1 Uber E..."
1,2,Premier League,The Premier League is the highest level of the...
2,3,La Liga,The Campeonato Nacional de Liga de Primera Div...
3,4,Fußball-Bundesliga,The Bundesliga sometimes referred to as the Fu...
4,5,Serie A,The Serie A officially known as Serie A TIM fo...


<h3>2. Dim Team</h3>

In [13]:
fbref_matchinfos.loc[fbref_matchinfos['Home_Team'] == 'Gladbach', 'Home_Team'] = "M'Gladbach"
fbref_matchinfos.loc[fbref_matchinfos['Away_Team'] == 'Gladbach', 'Away_Team'] = "M'Gladbach"

In [14]:
dim_team = pd.DataFrame({'Team_Key': [],
                          'Team_Name': [],
                          'Team_Description': []})

list_team_key = []
list_team_name = fbref_matchinfos['Home_Team'].unique().tolist()

for i in range(0,len(list_team_name)):
    list_team_key.append(i+1)

dim_team['Team_Key'] = list_team_key
dim_team['Team_Name'] = list_team_name

In [15]:
dim_team

,Team_Key,Team_Name,Team_Description
0,1,Monaco,NaN
1,2,Paris S-G,NaN
2,3,Lyon,NaN
3,4,Saint-Étienne,NaN
4,5,Metz,NaN
...,...,...,...
140,141,Spezia,NaN
141,142,Venezia,NaN
142,143,Salernitana,NaN
143,144,Monza,NaN


<h3>3. Dim Date</h3>

In [ ]:
dates_13_18 = pd.read_csv('gs://football_data_etl/football_data_extracted/output-basic/SampleDateDim_13_18.csv')
dates_19_24 = pd.read_csv('gs://football_data_etl/football_data_extracted/output-basic/SampleDateDim_19_24.csv')

In [17]:
dates = pd.concat([dates_13_18,dates_19_24], ignore_index=True)

In [18]:
dim_date = pd.DataFrame({'Date_Key': [],
                          'Full_Date': [],
                          'Day_Of_Week': [],
                          'Day_Num_In_Month': [],
                          'Day_Name': [],
                          'Month': [],
                          'Month_Name': [],
                          'Quarter': [],
                          'Year': []})

dim_date['Date_Key'] = dates['date key'].to_list()
dim_date['Full_Date'] = dates['full date'].to_list()
dim_date['Day_Of_Week'] = dates['day of week'].to_list()
dim_date['Day_Num_In_Month'] = dates['day num in month'].to_list()
dim_date['Day_Name'] = dates['day name'].to_list()
dim_date['Month'] = dates['month'].to_list()
dim_date['Month_Name'] = dates['month name'].to_list()
dim_date['Quarter'] = dates['quarter'].to_list()
dim_date['Year'] = dates['year'].to_list()

In [19]:
dim_date['Full_Date'] = pd.to_datetime(dim_date['Full_Date'])

In [20]:
dim_date.head(10)

,Date_Key,Full_Date,Day_Of_Week,Day_Num_In_Month,Day_Name,Month,Month_Name,Quarter,Year
0,20130101,2013-01-01,2,1,Tuesday,1,January,1,2013
1,20130102,2013-01-02,3,2,Wednesday,1,January,1,2013
2,20130103,2013-01-03,4,3,Thursday,1,January,1,2013
3,20130104,2013-01-04,5,4,Friday,1,January,1,2013
4,20130105,2013-01-05,6,5,Saturday,1,January,1,2013
5,20130106,2013-01-06,7,6,Sunday,1,January,1,2013
6,20130107,2013-01-07,1,7,Monday,1,January,1,2013
7,20130108,2013-01-08,2,8,Tuesday,1,January,1,2013
8,20130109,2013-01-09,3,9,Wednesday,1,January,1,2013
9,20130110,2013-01-10,4,10,Thursday,1,January,1,2013


<h3>4. Dim Match</h3>

In [21]:
unique_values = fbref_matchinfos.loc[~fbref_matchinfos['Home_Team'].isin(fbref_matchstats['Team']), 'Home_Team'].unique()
unique_values

array([], dtype=object)

In [ ]:
#fbref_matchinfos.loc[fbref_matchinfos['Home_Team'] == 'Gladbach', 'Home_Team'] = "M'Gladbach"
#fbref_matchstats[fbref_matchstats['Team'] == "Gladbach"]

In [22]:
dim_match = pd.DataFrame({'Match_Key': [],
                          'Match_Id': [],
                          'League': [],
                          'Season': [],
                          'Match_Week': [],
                          'Home_Team': [],
                          'Away_Team': [],
                          'Match_Date': [],
                          'Venue_Day': [],
                          'Venue_Time': [],
                          'Attendance': [],
                          'Stadium': [],
                          'Officials': [],
                          'Link': [],
                          'Home_Team_Formation': [],
                          'Away_Team_Formation': [],
                          'Result': []})

#'Home_Team_Manager': [],'Home_Team_Captain': [], Away_Team_Manager': [],'Away_Team_Captain': [],'Away_Team_Formation': []

list_match_key = []
list_match_id = fbref_matchinfos['Match_Id'].to_list()

for i in range(0,len(list_match_id)):
    list_match_key.append(i+1)

dim_match['Match_Key'] = list_match_key
dim_match['Match_Id'] = list_match_id
dim_match['Season'] = fbref_matchinfos['Season'].to_list()
dim_match['Match_Week'] = fbref_matchinfos['Match_Week'].to_list()

dim_match['Venue_Day'] = fbref_matchinfos['Venue_Day'].to_list()
dim_match['Venue_Time'] = fbref_matchinfos['Venue_Time'].to_list()
dim_match['Attendance'] = fbref_matchinfos['Attendance'].to_list()
dim_match['Stadium'] = fbref_matchinfos['Stadium'].to_list()
dim_match['Officials'] = fbref_matchinfos['Officials'].to_list()
dim_match['Link'] = fbref_matchinfos['Link'].to_list()

#
list_home_team_formation = []
list_away_team_formation = []
list_result = []

for i in range(0, len(fbref_matchinfos)):

    team_df = fbref_matchstats[(fbref_matchstats['Match_Id'] == fbref_matchinfos['Match_Id'].iloc[i]) &
                               (fbref_matchstats['Team'] == fbref_matchinfos['Home_Team'].iloc[i])]

    if (len(team_df) > 0):
        list_home_team_formation.append(team_df['Formation'].iloc[0])
        home_score = team_df['Score'].iloc[0]
    else:
        list_home_team_formation.append(np.nan)
        home_score = -1


    team_df = fbref_matchstats[(fbref_matchstats['Match_Id'] == fbref_matchinfos['Match_Id'].iloc[i]) &
                               (fbref_matchstats['Team'] == fbref_matchinfos['Away_Team'].iloc[i])]

    if (len(team_df) > 0):
        list_away_team_formation.append(team_df['Formation'].iloc[0])
        away_score = team_df['Score'].iloc[0]
    else:
        list_away_team_formation.append(np.nan)
        away_score = -1

    if(home_score == -1 or away_score == -1):
        list_result.append(np.nan)
    else:
        if(home_score > away_score):
            list_result.append("Win")
        elif (home_score == away_score):
            list_result.append("Draw")
        else:
            list_result.append("Lose")


dim_match['Home_Team_Formation'] = list_home_team_formation
dim_match['Away_Team_Formation'] = list_away_team_formation

dim_match['Result'] = list_result

#fk
list_league = []
list_home_team = []
list_away_team = []
list_match_date = []

for i in range(0, len(fbref_matchinfos)):
    print(i)

    team_df = dim_league[dim_league['League_Name'] == fbref_matchinfos['League'].iloc[i]]
    list_league.append(team_df['League_Key'].iloc[0])

    team_df = dim_team[dim_team['Team_Name'] == fbref_matchinfos['Home_Team'].iloc[i]]
    list_home_team.append(team_df['Team_Key'].iloc[0])

    team_df = dim_team[dim_team['Team_Name'] == fbref_matchinfos['Away_Team'].iloc[i]]
    list_away_team.append(team_df['Team_Key'].iloc[0])

    team_df = dim_date[dim_date['Full_Date'] == fbref_matchinfos['Match_Date'].iloc[i]]
    list_match_date.append(team_df['Date_Key'].iloc[0])


dim_match['League'] = list_league
dim_match['Home_Team'] = list_home_team
dim_match['Away_Team'] = list_away_team
dim_match['Match_Date'] = list_match_date

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
7708
7709
7710
7711
7712
7713
7714
7715
7716
7717
7718
7719
7720
7721
7722
7723
7724
7725
7726
7727
7728
7729
7730
7731
7732
7733
7734
7735
7736
7737
7738
7739
7740
7741
7742
7743
7744
7745
7746
7747
7748
7749
7750
7751
7752
7753
7754
7755
7756
7757
7758
7759
7760
7761
7762
7763
7764
7765
7766
7767
7768
7769
7770
7771
7772
7773
7774
7775
7776
7777
7778
7779
7780
7781
7782
7783
7784
7785
7786
7787
7788
7789
7790
7791
7792
7793
7794
7795
7796
7797
7798
7799
7800
7801
7802
7803
7804
7805
7806
7807
7808
7809
7810
7811
7812
7813
7814
7815
7816
7817
7818
7819
7820
7821
7822
7823
7824
7825
7826
7827
7828
7829
7830
7831
7832
7833
7834
7835
7836
7837
7838
7839
7840
7841
7842
7843
7844
7845
7846
7847
7848
7849
7850
7851
7852
7853
7854
7855
7856
7857
7858
7859
7860
7861
7862
7863
7864
7865
7866
7867
7868
7869
7870
7871
7872
7873
7874
7875
7876
7877
7878
7879
7880
7881
7882
7883
7884
7885
7886
7887
7888
7889
7890
7891
7892
7893
7894
7895
789

In [23]:
dim_match

,Match_Key,Match_Id,League,Season,Match_Week,Home_Team,Away_Team,Match_Date,Venue_Day,Venue_Time,Attendance,Stadium,Officials,Link,Home_Team_Formation,Away_Team_Formation,Result
0,1,a68e623d,1,2018,1,1,20,20170804,Fri,2024-07-02 20:45,13572.0,Stade Louis II,Clément Turpin,https://fbref.com/en/matches/a68e623d/Monaco-T...,4/4/2002,4/3/2003,Win
1,2,37f2c25f,1,2018,1,2,13,20170805,Sat,2024-07-02 17:15,46898.0,Parc des Princes,Mikael Lesage,https://fbref.com/en/matches/37f2c25f/Paris-Sa...,4/3/2003,5/4/2001,Win
2,3,4d28b63b,1,2018,1,3,19,20170805,Sat,2024-07-02 20:00,1979.0,Groupama Stadium,Ruddy Buquet,https://fbref.com/en/matches/4d28b63b/Lyon-Str...,4-2-3-1,4-1-4-1,Win
3,4,68b9eea2,1,2018,1,4,14,20170805,Sat,2024-07-02 20:00,25879.0,Stade Geoffroy-Guichard,François Letexier,https://fbref.com/en/matches/68b9eea2/Saint-Et...,4/3/2003,4-2-3-1,Win
4,5,7d2eb66d,1,2018,1,5,17,20170805,Sat,2024-07-02 20:00,14595.0,Stade Saint-Symphorien,Jérôme Miguelgorry,https://fbref.com/en/matches/7d2eb66d/Metz-Gui...,4-2-3-1,4/4/2002,Lose
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12703,12704,6635e569,5,2024,9,143,126,20231022,Sun,2024-07-02 15:00,17409.0,Stadio Arechi,Daniele Chiffi,https://fbref.com/en/matches/6635e569/Salernit...,4-3-2-1,4-3-1-2,Draw
12704,12705,045aca2a,5,2024,9,118,130,20231022,Sun,2024-07-02 18:00,14848.0,Gewiss Stadium,Livio Marinelli,https://fbref.com/en/matches/045aca2a/Atalanta...,3/4/2003,3/5/2002,Win
12705,12706,bea57463,5,2024,9,131,116,20231022,Sun,2024-07-02 20:45,75676.0,Stadio Giuseppe Meazza,Maurizio Mariani,https://fbref.com/en/matches/bea57463/Milan-Ju...,4-2-3-1,3/5/2002,Lose
12706,12707,cd802be7,5,2024,9,119,140,20231023,Mon,2024-07-02 18:30,20113.0,Bluenergy Stadium,Paride Tremolada,https://fbref.com/en/matches/cd802be7/Udinese-...,3/5/2002,4/3/2003,Draw


In [24]:
dim_match.isna().sum()

Match_Key                 0
Match_Id                  0
League                    0
Season                    0
Match_Week                0
Home_Team                 0
Away_Team                 0
Match_Date                0
Venue_Day                 0
Venue_Time               73
Attendance             2266
Stadium                   0
Officials               101
Link                      0
Home_Team_Formation     104
Away_Team_Formation     104
Result                  104
dtype: int64

<h3>5. Dim Player</h3>

In [ ]:
dim_player = pd.DataFrame({'Player_Key': [],
                          'Player_Id': [],
                          'Player_Name': [],
                          'Player_Full_Name': [],
                          'Birthdate': []})

list_player_key = []
list_player_id = sofifa_players_attr['Sofifa_Id'].unique().tolist()

for i in range(0,len(list_player_id)):
    list_player_key.append(i+1)

dim_player['Player_Key'] = list_player_key
dim_player['Player_Id'] = list_player_id

list_player_name = []
list_player_full_name = []
list_birthdate = []

for i in range(0, len(list_player_id)):
    print(i)

    temp_df = sofifa_players_attr[sofifa_players_attr['Sofifa_Id'] == list_player_id[i]]

    if (len(temp_df) > 0):
        list_player_name.append(temp_df['Player_Name'].iloc[0])
        list_player_full_name.append(temp_df['Player_Full_Name'].iloc[0])
        list_birthdate.append(temp_df['Birthday'].iloc[0])
    else:
        list_player_name.append(np.nan)
        list_player_full_name.append(np.nan)
        list_birthdate.append(np.nan)

dim_player['Player_Name'] = list_player_name
dim_player['Player_Full_Name'] = list_player_full_name
dim_player['Birthdate'] = list_birthdate

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
4180
4181
4182
4183
4184
4185
4186
4187
4188
4189
4190
4191
4192
4193
4194
4195
4196
4197
4198
4199
4200
4201
4202
4203
4204
4205
4206
4207
4208
4209
4210
4211
4212
4213
4214
4215
4216
4217
4218
4219
4220
4221
4222
4223
4224
4225
4226
4227
4228
4229
4230
4231
4232
4233
4234
4235
4236
4237
4238
4239
4240
4241
4242
4243
4244
4245
4246
4247
4248
4249
4250
4251
4252
4253
4254
4255
4256
4257
4258
4259
4260
4261
4262
4263
4264
4265
4266
4267
4268
4269
4270
4271
4272
4273
4274
4275
4276
4277
4278
4279
4280
4281
4282
4283
4284
4285
4286
4287
4288
4289
4290
4291
4292
4293
4294
4295
4296
4297
4298
4299
4300
4301
4302
4303
4304
4305
4306
4307
4308
4309
4310
4311
4312
4313
4314
4315
4316
4317
4318
4319
4320
4321
4322
4323
4324
4325
4326
4327
4328
4329
4330
4331
4332
4333
4334
4335
4336
4337
4338
4339
4340
4341
4342
4343
4344
4345
4346
4347
4348
4349
4350
4351
4352
4353
4354
4355
4356
4357
4358
4359
4360
4361
4362
4363
4364
4365
4366
4367
436

In [ ]:
dim_player

,Player_Key,Player_Id,Player_Name,Player_Full_Name,Birthdate
0,1,223597.0,R. Aguilar,Ruben Aguilar,1993-04-26
1,2,224069.0,K. Toko-Ekambi,Karl Brillant Toko Ekambi,1992-09-14
2,3,225085.0,J. Bamba,Jonathan Bamba,1996-03-26
3,4,171791.0,José Fonte,José Miguel da Rocha Fonte,1983-12-22
4,5,241727.0,P. Rajković,Predrag Rajković,1995-10-31
...,...,...,...,...,...
9175,9176,256948.0,C. Tzolis,Christos Tzolis,2002-01-30
9176,9177,236457.0,D. Giannoulis,Dimitris Giannoulis,1995-10-17
9177,9178,223154.0,O. Tufan,Ozan Tufan,1995-03-23
9178,9179,260142.0,R. Alebiosu,Ryan Alebiousu,2001-12-17


In [ ]:
dim_player.isna().sum()

Player_Key          0
Player_Id           1
Player_Name         1
Player_Full_Name    1
Birthdate           1
dtype: int64

<h3>6. Dim Player Version</h3>

In [ ]:
unique_values = fbref_matchinfos.loc[~fbref_matchinfos['Home_Team'].isin(sofifa_players_attr['Club']), 'Home_Team'].unique()
unique_values

array(['Hamburger SV', 'Düsseldorf', 'Paderborn 07', 'Arminia',
       'Greuther Fürth'], dtype=object)

In [ ]:
len(sofifa_players_attr)

291047

In [ ]:
dim_player_version = pd.DataFrame({'Player_Version_Key': [],
                          'Player': [],
                          'Update_Date': [],
                          'Team_Club': [],
                          'Team_National': [],
                          'Height': [],
                          'Weight': [],
                          'All_Positions': [], 'Attacking_Work_Rate': [], 'Body_Type': [],
                          'Defensive_Work_Rate': [], 'Preferred_Foot': [],
                          'Team_Club_Contract': [], 'Team_Club_Joined': [], 'Team_Club_Kitnum': [],
                          'Team_Club_Loaned_From': [], 'Team_National_Positions': [],
                          'Team_National_Rating': [], 'Specialities': []})

list_player_version_key = []
for i in range(0,len(sofifa_players_attr)):
    list_player_version_key.append(i+1)

dim_player_version['Player_Version_Key'] = list_player_version_key
dim_player_version['Team_National'] = sofifa_players_attr['National_Team'].to_list()
dim_player_version['Height'] = sofifa_players_attr['Height'].to_list()
dim_player_version['Weight'] = sofifa_players_attr['Weight'].to_list()
dim_player_version['All_Positions'] = sofifa_players_attr['All_Positions'].to_list()
dim_player_version['Attacking_Work_Rate'] = sofifa_players_attr['Attacking_Work_Rate'].to_list()
dim_player_version['Body_Type'] = sofifa_players_attr['Body_Type'].to_list()
dim_player_version['Defensive_Work_Rate'] = sofifa_players_attr['Defensive_Work_Rate'].to_list()
dim_player_version['Preferred_Foot'] = sofifa_players_attr['Preferred_Foot'].to_list()
dim_player_version['Team_Club_Contract'] = sofifa_players_attr['Club_Contract'].to_list()
dim_player_version['Team_Club_Joined'] = sofifa_players_attr['Club_Joined'].to_list()
dim_player_version['Team_Club_Kitnum'] = sofifa_players_attr['Club_Kitnum'].to_list()
dim_player_version['Team_Club_Loaned_From'] = sofifa_players_attr['Club_Loaned_From'].to_list()
dim_player_version['Team_National_Positions'] = sofifa_players_attr['National_Team_Position'].to_list()
dim_player_version['Team_National_Rating'] = sofifa_players_attr['National_Team_Rating'].to_list()
dim_player_version['Specialities'] = sofifa_players_attr['Specialities'].to_list()


#fk
list_player = []
list_update_date = []
list_team_club = []

for i in range(0, len(sofifa_players_attr)):

    print(i)

    temp_df = dim_player[dim_player['Player_Id'] == sofifa_players_attr['Sofifa_Id'].iloc[i]]
    if len(temp_df) > 0:
        list_player.append(temp_df['Player_Key'].iloc[0])
    else:
        list_player.append(np.nan)

    temp_df = dim_date[dim_date['Full_Date'] == sofifa_players_attr['Update_Date'].iloc[i]]
    if len(temp_df) > 0:
        list_update_date.append(temp_df['Date_Key'].iloc[0])
    else:
        list_update_date.append(np.nan)


    temp_df = dim_team[dim_team['Team_Name'] == sofifa_players_attr['Club'].iloc[i]]
    if len(temp_df) > 0:
        list_team_club.append(temp_df['Team_Key'].iloc[0])
    else:
        list_team_club.append(np.nan)

dim_player_version['Player'] = list_player
dim_player_version['Update_Date'] = list_update_date
dim_player_version['Team_Club'] = list_team_club


Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
286047
286048
286049
286050
286051
286052
286053
286054
286055
286056
286057
286058
286059
286060
286061
286062
286063
286064
286065
286066
286067
286068
286069
286070
286071
286072
286073
286074
286075
286076
286077
286078
286079
286080
286081
286082
286083
286084
286085
286086
286087
286088
286089
286090
286091
286092
286093
286094
286095
286096
286097
286098
286099
286100
286101
286102
286103
286104
286105
286106
286107
286108
286109
286110
286111
286112
286113
286114
286115
286116
286117
286118
286119
286120
286121
286122
286123
286124
286125
286126
286127
286128
286129
286130
286131
286132
286133
286134
286135
286136
286137
286138
286139
286140
286141
286142
286143
286144
286145
286146
286147
286148
286149
286150
286151
286152
286153
286154
286155
286156
286157
286158
286159
286160
286161
286162
286163
286164
286165
286166
286167
286168
286169
286170
286171
286172
286173
286174
286175
286176
286177
286178
286179
286180
28618

In [ ]:
dim_player_version

,Player_Version_Key,Player,Update_Date,Team_Club,Team_National,Height,Weight,All_Positions,Attacking_Work_Rate,Body_Type,Defensive_Work_Rate,Preferred_Foot,Team_Club_Contract,Team_Club_Joined,Team_Club_Kitnum,Team_Club_Loaned_From,Team_National_Positions,Team_National_Rating,Specialities
0,1,1.0,20200923.0,1.0,NaN,172,71.0,RB RWB,High,Normal (170-185),Medium,Right,2024,2019-08-06,26,NaN,NaN,NaN,NaN
1,2,2.0,20200923.0,3.0,Cameroon,185,78.0,ST LM,High,Normal (185+),Medium,Right,2024,2020-06-08,21,NaN,SUB,75.0,NaN
2,3,3.0,20200923.0,8.0,NaN,175,70.0,LM,High,Normal (170-185),High,Right,2023,2018-07-02,14,NaN,NaN,NaN,NaN
3,4,4.0,20200923.0,8.0,Portugal,191,81.0,CB,Medium,Lean (185+),High,Right,2021,2018-07-15,6,NaN,SUB,83.0,NaN
4,5,5.0,20200923.0,21.0,NaN,191,85.0,GK,Medium,Normal (185+),Medium,Right,2023,2019-06-21,1,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
291042,291043,5025.0,20230705.0,38.0,NaN,175,71.0,RW,High,Unique,Medium,Left,2025,2017-07-01,11,NaN,NaN,NaN,#Dribbler#Acrobat#Clinical finisher#Complete f...
291043,291044,5018.0,20230705.0,48.0,Norway,195,94.0,ST,High,Unique,Medium,Left,2027,2022-07-01,9,NaN,ST,76.0,#Aerial threat#Distance shooter#Strength#Clini...
291044,291045,5031.0,20230705.0,48.0,Belgium,181,75.0,CM CAM,High,Unique,Medium,Right,2025,2015-08-30,17,NaN,RF,80.0,#Dribbler#Playmaker #Distance shooter#Crosser#...
291045,291046,6798.0,20230605.0,38.0,NaN,178,73.0,LB,High,Normal (170-185),Medium,Left,2024,2021-08-30,77,NaN,NaN,NaN,NaN


In [ ]:
dim_player_version.isna().sum()

Player_Version_Key              0
Player                         19
Update_Date                    19
Team_Club                    3730
Team_National              247723
Height                          0
Weight                         19
All_Positions                   0
Attacking_Work_Rate             0
Body_Type                       0
Defensive_Work_Rate             0
Preferred_Foot                 19
Team_Club_Contract              0
Team_Club_Joined            22924
Team_Club_Kitnum                0
Team_Club_Loaned_From      268123
Team_National_Positions    247723
Team_National_Rating       247723
Specialities               246361
dtype: int64

<h3>7. Dim MatchSquad</h3>

In [ ]:
#sofifa
sofifa_players = pd.read_csv('gs://football_data_etl/football_data_extracted/output-basic/sofifa_players_attr.csv/part-00000-0474179e-867f-4f31-9d30-40c396f830f5-c000.csv',
                                  on_bad_lines="skip")
sofifa_players['Birthday'] = pd.to_datetime(sofifa_players['Birthday'])
sofifa_players['Birthday'] = sofifa_players['Birthday'].dt.strftime('%Y-%m-%d')
sofifa_players['Club_Joined'] = pd.to_datetime(sofifa_players['Club_Joined'])
sofifa_players['Club_Joined'] = sofifa_players['Club_Joined'].dt.strftime('%Y-%m-%d')
sofifa_players['Update_Date'] = pd.to_datetime(sofifa_players['Update_Date'])
sofifa_players['Update_Date'] = sofifa_players['Update_Date'].dt.strftime('%Y-%m-%d')

/tmp/ipykernel_270595/3872846029.py:2: DtypeWarning: Columns (45) have mixed types. Specify dtype option on import or set low_memory=False.
  sofifa_players = pd.read_csv('gs://football-data-etl/football-data-extracted/output-basic/sofifa_players_attr.csv/part-00000-c052c0ee-65cd-4820-8e09-1e44e809dcc5-c000.csv',


In [ ]:
different_team = set(fbref_matchsquad['Team']) - set(sofifa_players['Club'])
different_team

{'Arminia',
 'Cardiff City',
 'Düsseldorf',
 'Greuther Fürth',
 'Hamburger SV',
 'Hannover 96',
 'Huddersfield',
 'Norwich City',
 'Nürnberg',
 'Paderborn 07',
 'Stoke City',
 'Swansea City',
 'Watford',
 'West Brom'}

Merge

In [ ]:
fbref_matchsquad_new = pd.merge(fbref_matchsquad, fbref_matchinfos, on='Match_Id')
fbref_matchsquad_new = fbref_matchsquad_new.loc[:,['Match_Id', 'Match_Date', 'Team',
                                                             'Player_Name', 'Player_Kitnum', 'Is_Sub', 'Season']]
fbref_matchsquad_new.head(10)

,Match_Id,Match_Date,Team,Player_Name,Player_Kitnum,Is_Sub,Season
0,39b738ac,2017-11-24,Saint-Étienne,Stéphane Ruffier,16,No,2018
1,39b738ac,2017-11-24,Saint-Étienne,Kévin Théophile-Catherine,2,No,2018
2,39b738ac,2017-11-24,Saint-Étienne,Vincent Pajot,5,No,2018
3,39b738ac,2017-11-24,Saint-Étienne,Assane Dioussé,8,No,2018
4,39b738ac,2017-11-24,Saint-Étienne,Jonathan Bamba,14,No,2018
5,39b738ac,2017-11-24,Saint-Étienne,Florentin Pogba,19,No,2018
6,39b738ac,2017-11-24,Saint-Étienne,Hernani,20,No,2018
7,39b738ac,2017-11-24,Saint-Étienne,Kévin Monnet-Paquet,22,No,2018
8,39b738ac,2017-11-24,Saint-Étienne,Alexander Søderlund,23,No,2018
9,39b738ac,2017-11-24,Saint-Étienne,Loïc Perrin,24,No,2018


In [ ]:
#remove season 2017-2018
fbref_matchsquad_new = fbref_matchsquad_new[fbref_matchsquad_new['Season'] != 2018]
fbref_matchsquad_new = fbref_matchsquad_new.drop('Season', axis=1)
fbref_matchsquad_new.count()

Match_Id         428843
Match_Date       428843
Team             428843
Player_Name      428843
Player_Kitnum    428843
Is_Sub           428843
dtype: int64

Sắp xếp Sofifa_playes theo ngày cập nhật Update_Date giảm dần và tăng dần

In [ ]:
sofifa_players_desc_update_date = sofifa_players.sort_values(by = 'Update_Date', ascending=False)
sofifa_players_asc_update_date = sofifa_players.sort_values(by = 'Update_Date', ascending=True)

* function() ClosestUpdateDate: trả về ngày Update_Date gần nhất cho Match_Date (ngày diễn ra match)
* function() SofifaPlayersEqualUpdateDate: trả về dataframe tất cả các sofifa player với 1 Update_Date

In [ ]:
#return closest Update_Date: date-time
def ClosestUpdateDate(sofifa_players_desc_update_date, match_date):
    closest_update_date = sofifa_players_desc_update_date[sofifa_players_desc_update_date['Update_Date']
                                                        <= match_date]['Update_Date'].iloc[0]
    return closest_update_date

#return dataframe
def SofifaPlayersEqualUpdateDate(sofifa_players_desc_update_date, closest_update_date):
    sofia_players_one_update_version = sofifa_players_desc_update_date[sofifa_players_desc_update_date['Update_Date']
                                                                    == closest_update_date]
    return sofia_players_one_update_version

* function() SofifaPlayer: trả về player với id, position và overall rating ứng với mỗi match
* function() SofifaPlayer_ByName: trả về player theo name

In [ ]:
#return dataframe:
def SofifaPlayer(sofifa_update_version, team_fbref, player_kitnum_fbref):
    player = sofifa_update_version[(sofifa_update_version['Club'] == team_fbref) &
                                   (sofifa_update_version['Club_Kitnum'] == player_kitnum_fbref)][['Sofifa_Id', 'All_Positions',
                                                                                                   'Overall_Rating', 'Update_Date']]
    return player

#return dataframe:
def SofifaPlayer_ByName(sofifa_update_version, player_name):
    player = sofifa_update_version[sofifa_update_version['Player_Name']
                                            == player_name][['Sofifa_Id',
                                                             'All_Positions',
                                                             'Overall_Rating',
                                                             'Update_Date']]
    return player

- function() PLayerNotFound: kiểm tra trường hợp player trong match không có trong phiên bản sofifa gần nhất

In [ ]:
#return player_id: integer
def PLayerNotFound(sofifa_players_asc_update_date,closest_update_date, player_name):

    player = pd.DataFrame()

    dates_last = sofifa_players_asc_update_date[sofifa_players_asc_update_date['Update_Date']
                                                            > closest_update_date]['Update_Date'].unique()
    if(len(dates_last)>0):
        for k in range(0,len(dates_last)):
            sofia_players_more_update_date = sofifa_players_asc_update_date[sofifa_players_asc_update_date['Update_Date']
                                                                        == dates_last[k]]

            player = SofifaPlayer_ByName(sofia_players_more_update_date, player_name)

            if not player.empty:
                break

    return player

Cập nhật thêm cho MatchSquad với sofifa_id player, vị trí, overall rating và phiên bản player

In [ ]:
new_attrs = pd.DataFrame({'Sofifa_Id': [],
                          'Player_Position': [],
                          'Player_Overall_Rating': [],
                          'Player_Update_Date': []})
list_sofifa_id = []
list_player_position = []
list_player_overall_rating = []
list_update_date = []

Xác định ứng với mỗi match, tại thời điểm Update_Date của player gần nhất, với mỗi Team - đội bóng và Player_Kitnum - số áo player trong đội ta có thể tìm được player tương ứng trong Sofifa

In [ ]:
for i in range(0, len(fbref_matchsquad_new)):
    print(i)
    #
    match_date_i = fbref_matchsquad_new.iloc[i, fbref_matchsquad_new.columns.get_loc('Match_Date')]

    closest_update_date_i = ClosestUpdateDate(sofifa_players_desc_update_date, match_date_i)

    # Sofifa_Id attribute
    # ** team **
    team_fbref_i = fbref_matchsquad_new.iloc[i, fbref_matchsquad_new.columns.get_loc('Team')]
    # ** player_kit_num **
    player_kitnum_fbref_i = fbref_matchsquad_new.iloc[i, fbref_matchsquad_new.columns.get_loc('Player_Kitnum')]

    sofia_players_this_update_version = SofifaPlayersEqualUpdateDate(sofifa_players_desc_update_date, closest_update_date_i)

    player = SofifaPlayer(sofia_players_this_update_version, team_fbref_i, player_kitnum_fbref_i)

    player_id = np.nan
    player_position = np.nan
    player_overall_rating = np.nan
    player_update_date = np.nan

    #if player.empty:
        #player = PLayerNotFound(sofifa_players_asc_update_date,closest_update_date_i, team_fbref_i, player_kitnum_fbref_i)

    if not player.empty:
        player_id = player['Sofifa_Id'].iloc[0]
        player_position = player['All_Positions'].iloc[0]#.split(' ')[0]
        player_overall_rating = player['Overall_Rating'].iloc[0]
        player_update_date = player['Update_Date'].iloc[0]


    list_sofifa_id.append(player_id)

    #Player_Position attribute
    list_player_position.append(player_position)

    #Player_Overall_Rating attribute
    list_player_overall_rating.append(player_overall_rating)

    #Player_Update_Date attribute
    list_update_date.append(player_update_date)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

1861
1862
1863
1864
1865
1866
1867
1868
1869
1870
1871
1872
1873
1874
1875
1876
1877
1878
1879
1880
1881
1882
1883
1884
1885
1886
1887
1888
1889
1890
1891
1892
1893
1894
1895
1896
1897
1898
1899
1900
1901
1902
1903
1904
1905
1906
1907
1908
1909
1910
1911
1912
1913
1914
1915
1916
1917
1918
1919
1920
1921
1922
1923
1924
1925
1926
1927
1928
1929
1930
1931
1932
1933
1934
1935
1936
1937
1938
1939
1940
1941
1942
1943
1944
1945
1946
1947
1948
1949
1950
1951
1952
1953
1954
1955
1956
1957
1958
1959
1960
1961
1962
1963
1964
1965
1966
1967
1968
1969
1970
1971
1972
1973
1974
1975
1976
1977
1978
1979
1980
1981
1982
1983
1984
1985
1986
1987
1988
1989
1990
1991
1992
1993
1994
1995
1996
1997
1998
1999
2000
2001
2002
2003
2004
2005
2006
2007
2008
2009
2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021
2022
2023
2024
2025
2026
2027
2028
2029
2030
2031
2032
2033
2034
2035
2036
2037
2038
2039
2040
2041
2042
2043
2044
2045
2046
2047
2048
2049
2050
2051
2052
2053
2054
2055
2056
2057
2058
2059
2060


3502
3503
3504
3505
3506
3507
3508
3509
3510
3511
3512
3513
3514
3515
3516
3517
3518
3519
3520
3521
3522
3523
3524
3525
3526
3527
3528
3529
3530
3531
3532
3533
3534
3535
3536
3537
3538
3539
3540
3541
3542
3543
3544
3545
3546
3547
3548
3549
3550
3551
3552
3553
3554
3555
3556
3557
3558
3559
3560
3561
3562
3563
3564
3565
3566
3567
3568
3569
3570
3571
3572
3573
3574
3575
3576
3577
3578
3579
3580
3581
3582
3583
3584
3585
3586
3587
3588
3589
3590
3591
3592
3593
3594
3595
3596
3597
3598
3599
3600
3601
3602
3603
3604
3605
3606
3607
3608
3609
3610
3611
3612
3613
3614
3615
3616
3617
3618
3619
3620
3621
3622
3623
3624
3625
3626
3627
3628
3629
3630
3631
3632
3633
3634
3635
3636
3637
3638
3639
3640
3641
3642
3643
3644
3645
3646
3647
3648
3649
3650
3651
3652
3653
3654
3655
3656
3657
3658
3659
3660
3661
3662
3663
3664
3665
3666
3667
3668
3669
3670
3671
3672
3673
3674
3675
3676
3677
3678
3679
3680
3681
3682
3683
3684
3685
3686
3687
3688
3689
3690
3691
3692
3693
3694
3695
3696
3697
3698
3699
3700
3701


5143
5144
5145
5146
5147
5148
5149
5150
5151
5152
5153
5154
5155
5156
5157
5158
5159
5160
5161
5162
5163
5164
5165
5166
5167
5168
5169
5170
5171
5172
5173
5174
5175
5176
5177
5178
5179
5180
5181
5182
5183
5184
5185
5186
5187
5188
5189
5190
5191
5192
5193
5194
5195
5196
5197
5198
5199
5200
5201
5202
5203
5204
5205
5206
5207
5208
5209
5210
5211
5212
5213
5214
5215
5216
5217
5218
5219
5220
5221
5222
5223
5224
5225
5226
5227
5228
5229
5230
5231
5232
5233
5234
5235
5236
5237
5238
5239
5240
5241
5242
5243
5244
5245
5246
5247
5248
5249
5250
5251
5252
5253
5254
5255
5256
5257
5258
5259
5260
5261
5262
5263
5264
5265
5266
5267
5268
5269
5270
5271
5272
5273
5274
5275
5276
5277
5278
5279
5280
5281
5282
5283
5284
5285
5286
5287
5288
5289
5290
5291
5292
5293
5294
5295
5296
5297
5298
5299
5300
5301
5302
5303
5304
5305
5306
5307
5308
5309
5310
5311
5312
5313
5314
5315
5316
5317
5318
5319
5320
5321
5322
5323
5324
5325
5326
5327
5328
5329
5330
5331
5332
5333
5334
5335
5336
5337
5338
5339
5340
5341
5342


6784
6785
6786
6787
6788
6789
6790
6791
6792
6793
6794
6795
6796
6797
6798
6799
6800
6801
6802
6803
6804
6805
6806
6807
6808
6809
6810
6811
6812
6813
6814
6815
6816
6817
6818
6819
6820
6821
6822
6823
6824
6825
6826
6827
6828
6829
6830
6831
6832
6833
6834
6835
6836
6837
6838
6839
6840
6841
6842
6843
6844
6845
6846
6847
6848
6849
6850
6851
6852
6853
6854
6855
6856
6857
6858
6859
6860
6861
6862
6863
6864
6865
6866
6867
6868
6869
6870
6871
6872
6873
6874
6875
6876
6877
6878
6879
6880
6881
6882
6883
6884
6885
6886
6887
6888
6889
6890
6891
6892
6893
6894
6895
6896
6897
6898
6899
6900
6901
6902
6903
6904
6905
6906
6907
6908
6909
6910
6911
6912
6913
6914
6915
6916
6917
6918
6919
6920
6921
6922
6923
6924
6925
6926
6927
6928
6929
6930
6931
6932
6933
6934
6935
6936
6937
6938
6939
6940
6941
6942
6943
6944
6945
6946
6947
6948
6949
6950
6951
6952
6953
6954
6955
6956
6957
6958
6959
6960
6961
6962
6963
6964
6965
6966
6967
6968
6969
6970
6971
6972
6973
6974
6975
6976
6977
6978
6979
6980
6981
6982
6983


...

12788
12789
12790
12791
12792
12793
12794
12795
12796
12797
12798
12799
12800
12801
12802
12803
12804
12805
12806
12807
12808
12809
12810
12811
12812
12813
12814
12815
12816
12817
12818
12819
12820
12821
12822
12823
12824
12825
12826
12827
12828
12829
12830
12831
12832
12833
12834
12835
12836
12837
12838
12839
12840
12841
12842
12843
12844
12845
12846
12847
12848
12849
12850
12851
12852
12853
12854
12855
12856
12857
12858
12859
12860
12861
12862
12863
12864
12865
12866
12867
12868
12869
12870
12871
12872
12873
12874
12875
12876
12877
12878
12879
12880
12881
12882
12883
12884
12885
12886
12887
12888
12889
12890
12891
12892
12893
12894
12895
12896
12897
12898
12899
12900
12901
12902
12903
12904
12905
12906
12907
12908
12909
12910
12911
12912
12913
12914
12915
12916
12917
12918
12919
12920
12921
12922
12923
12924
12925
12926
12927
12928
12929
12930
12931
12932
12933
12934
12935
12936
12937
12938
12939
12940
12941
12942
12943
12944
12945
12946
12947
12948
12949
12950
12951
12952
12953
1295

14155
14156
14157
14158
14159
14160
14161
14162
14163
14164
14165
14166
14167
14168
14169
14170
14171
14172
14173
14174
14175
14176
14177
14178
14179
14180
14181
14182
14183
14184
14185
14186
14187
14188
14189
14190
14191
14192
14193
14194
14195
14196
14197
14198
14199
14200
14201
14202
14203
14204
14205
14206
14207
14208
14209
14210
14211
14212
14213
14214
14215
14216
14217
14218
14219
14220
14221
14222
14223
14224
14225
14226
14227
14228
14229
14230
14231
14232
14233
14234
14235
14236
14237
14238
14239
14240
14241
14242
14243
14244
14245
14246
14247
14248
14249
14250
14251
14252
14253
14254
14255
14256
14257
14258
14259
14260
14261
14262
14263
14264
14265
14266
14267
14268
14269
14270
14271
14272
14273
14274
14275
14276
14277
14278
14279
14280
14281
14282
14283
14284
14285
14286
14287
14288
14289
14290
14291
14292
14293
14294
14295
14296
14297
14298
14299
14300
14301
14302
14303
14304
14305
14306
14307
14308
14309
14310
14311
14312
14313
14314
14315
14316
14317
14318
14319
14320
1432

15522
15523
15524
15525
15526
15527
15528
15529
15530
15531
15532
15533
15534
15535
15536
15537
15538
15539
15540
15541
15542
15543
15544
15545
15546
15547
15548
15549
15550
15551
15552
15553
15554
15555
15556
15557
15558
15559
15560
15561
15562
15563
15564
15565
15566
15567
15568
15569
15570
15571
15572
15573
15574
15575
15576
15577
15578
15579
15580
15581
15582
15583
15584
15585
15586
15587
15588
15589
15590
15591
15592
15593
15594
15595
15596
15597
15598
15599
15600
15601
15602
15603
15604
15605
15606
15607
15608
15609
15610
15611
15612
15613
15614
15615
15616
15617
15618
15619
15620
15621
15622
15623
15624
15625
15626
15627
15628
15629
15630
15631
15632
15633
15634
15635
15636
15637
15638
15639
15640
15641
15642
15643
15644
15645
15646
15647
15648
15649
15650
15651
15652
15653
15654
15655
15656
15657
15658
15659
15660
15661
15662
15663
15664
15665
15666
15667
15668
15669
15670
15671
15672
15673
15674
15675
15676
15677
15678
15679
15680
15681
15682
15683
15684
15685
15686
15687
1568

16889
16890
16891
16892
16893
16894
16895
16896
16897
16898
16899
16900
16901
16902
16903
16904
16905
16906
16907
16908
16909
16910
16911
16912
16913
16914
16915
16916
16917
16918
16919
16920
16921
16922
16923
16924
16925
16926
16927
16928
16929
16930
16931
16932
16933
16934
16935
16936
16937
16938
16939
16940
16941
16942
16943
16944
16945
16946
16947
16948
16949
16950
16951
16952
16953
16954
16955
16956
16957
16958
16959
16960
16961
16962
16963
16964
16965
16966
16967
16968
16969
16970
16971
16972
16973
16974
16975
16976
16977
16978
16979
16980
16981
16982
16983
16984
16985
16986
16987
16988
16989
16990
16991
16992
16993
16994
16995
16996
16997
16998
16999
17000
17001
17002
17003
17004
17005
17006
17007
17008
17009
17010
17011
17012
17013
17014
17015
17016
17017
17018
17019
17020
17021
17022
17023
17024
17025
17026
17027
17028
17029
17030
17031
17032
17033
17034
17035
17036
17037
17038
17039
17040
17041
17042
17043
17044
17045
17046
17047
17048
17049
17050
17051
17052
17053
17054
1705

18256
18257
18258
18259
18260
18261
18262
18263
18264
18265
18266
18267
18268
18269
18270
18271
18272
18273
18274
18275
18276
18277
18278
18279
18280
18281
18282
18283
18284
18285
18286
18287
18288
18289
18290
18291
18292
18293
18294
18295
18296
18297
18298
18299
18300
18301
18302
18303
18304
18305
18306
18307
18308
18309
18310
18311
18312
18313
18314
18315
18316
18317
18318
18319
18320
18321
18322
18323
18324
18325
18326
18327
18328
18329
18330
18331
18332
18333
18334
18335
18336
18337
18338
18339
18340
18341
18342
18343
18344
18345
18346
18347
18348
18349
18350
18351
18352
18353
18354
18355
18356
18357
18358
18359
18360
18361
18362
18363
18364


In [ ]:
new_attrs['Sofifa_Id'] = list_sofifa_id
new_attrs['Player_Position'] = list_player_position
new_attrs['Player_Overall_Rating'] = list_player_overall_rating
new_attrs['Player_Update_Date'] = list_update_date

In [ ]:
fbref_matchsquad_new1.reset_index(drop=True, inplace=True)
new_attrs.reset_index(drop=True, inplace=True)

matchsquad_players = pd.concat([fbref_matchsquad_new1, new_attrs], axis=1)

Existing files

In [ ]:
#sofifa
matchsquad_bundesliga = pd.read_csv('gs://football_data_etl/football_data_extracted/output-basic/matchsquad_players/matchsquad_players_bundesliga.csv')
matchsquad_laliga = pd.read_csv('gs://football_data_etl/football_data_extracted/output-basic/matchsquad_players/matchsquad_players_laliga.csv')
matchsquad_ligue1 = pd.read_csv('gs://football_data_etl/football_data_extracted/output-basic/matchsquad_players/matchsquad_players_ligue1.csv')
matchsquad_premierleague = pd.read_csv('gs://football_data_etl/football_data_extracted/output-basic/matchsquad_players/matchsquad_players_premier_league_new.csv')
matchsquad_seriea = pd.read_csv('gs://football_data_etl/football_data_extracted/output-basic/matchsquad_players/matchsquad_players_serie_a.csv')

In [26]:
merge = pd.concat([matchsquad_bundesliga, matchsquad_laliga,
                   matchsquad_ligue1, matchsquad_premierleague, matchsquad_seriea], axis=0)
merge['Match_Date'] = pd.to_datetime(merge['Match_Date'])
merge['Player_Update_Date'] = pd.to_datetime(merge['Player_Update_Date'])

In [27]:
merge

,Match_Id,Match_Date,Team,Player_Name,Player_Kitnum,Is_Sub,Sofifa_Id,Player_Position,Player_Overall_Rating,Player_Update_Date
0,3adcdef9,2018-09-26,M'Gladbach,Yann Sommer,1,No,177683.0,GK,82.0,2018-09-20
1,3adcdef9,2018-09-26,M'Gladbach,Christoph Kramer,6,No,204024.0,CDM CM,81.0,2018-09-20
2,3adcdef9,2018-09-26,M'Gladbach,Patrick Herrmann,7,No,198077.0,RM,77.0,2018-09-20
3,3adcdef9,2018-09-26,M'Gladbach,Thorgan Hazard,10,No,203486.0,LM RM CF,81.0,2018-09-20
4,3adcdef9,2018-09-26,M'Gladbach,Alassane Pléa,14,No,206467.0,ST RW,80.0,2018-09-20
...,...,...,...,...,...,...,...,...,...,...
103038,f6279636,2024-05-12,Empoli,Alberto Cerri,17,Yes,213829.0,ST,68.0,2024-04-09
103039,f6279636,2024-05-12,Empoli,Jacopo Fazzini,21,Yes,266596.0,CM CAM,67.0,2024-04-09
103040,f6279636,2024-05-12,Empoli,Mattia Destro,23,Yes,193338.0,ST,70.0,2024-04-09
103041,f6279636,2024-05-12,Empoli,Szymon Żurkowski,27,Yes,239732.0,CM CAM CF,72.0,2024-04-09


In [28]:
different_team = set(dim_team['Team_Name']) - set(merge['Team'])
different_team

{'Hamburger SV', 'La Coruña', 'Málaga', 'Stoke City', 'Swansea City'}

Dim Match Squad

In [ ]:
dim_match_squad = pd.DataFrame({'Match_Squad_Key': [],
                                'Match': [],
                                'Team': [],
                                'Player_Version': [],
                                'Sub': []})

list_match_squad_key = []
list_match = []
list_team = []
list_player_version = []

list_position = []
list_age = []

for i in range(0,len(merge)):
    print(i)
    list_match_squad_key.append(i+1)

    temp_df = dim_match[dim_match['Match_Id'] == merge['Match_Id'].iloc[i]]

    if (len(temp_df) > 0):
        list_match.append(temp_df['Match_Key'].iloc[0])
    else:
        list_match.append(np.nan)

    temp_df = dim_team[dim_team['Team_Name'] == merge['Team'].iloc[i]]
    if (len(temp_df) > 0):
        list_team.append(temp_df['Team_Key'].iloc[0])
    else:
        list_team.append(np.nan)

    if(~np.isnan(merge['Sofifa_Id'].iloc[i])):



        player_df = dim_player[dim_player['Player_Id'] == merge['Sofifa_Id'].iloc[i]]
        update_date_df = dim_date[dim_date['Full_Date'] == merge['Player_Update_Date'].iloc[i]]

        if (len(player_df) > 0):
            player_key = player_df['Player_Key'].iloc[0]
        else:
            player_key = np.nan

        if (len(update_date_df) > 0):
            update_date_key = update_date_df['Date_Key'].iloc[0]
        else:
            update_date_key = np.nan


        temp_df = dim_player_version[(dim_player_version['Player'] == player_key) &
                                 (dim_player_version['Update_Date'] == update_date_key)]

        if (len(temp_df) > 0):
            list_player_version.append(temp_df['Player_Version_Key'].iloc[0])
        else:
            list_player_version.append(np.nan)

    else:
        list_player_version.append(np.nan)

dim_match_squad['Match_Squad_Key'] = list_match_squad_key
dim_match_squad['Match'] = list_match
dim_match_squad['Team'] = list_team
dim_match_squad['Player_Version'] = list_player_version
dim_match_squad['Sub'] = merge['Is_Sub'].to_list()

#continue

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

ERROR:gcsfs:_request out of retries on exception: ("Failed to retrieve http://metadata.google.internal/computeMetadata/v1/instance/service-accounts/default/?recursive=true from the Google Compute Engine metadata service. Status: 404 Response:\nb''", <google.auth.transport.requests._Response object at 0x78c4b758ba00>)
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/google/auth/compute_engine/credentials.py", line 128, in refresh
    self._retrieve_info(request)
  File "/usr/local/lib/python3.10/dist-packages/google/auth/compute_engine/credentials.py", line 101, in _retrieve_info
    info = _metadata.get_service_account_info(
  File "/usr/local/lib/python3.10/dist-packages/google/auth/compute_engine/_metadata.py", line 323, in get_service_account_info
    return get(request, path, params={"recursive": "true"})
  File "/usr/local/lib/python3.10/dist-packages/google/auth/compute_engine/_metadata.py", line 248, in get
    raise exceptions.TransportError(
g

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
433993
433994
433995
433996
433997
433998
433999
434000
434001
434002
434003
434004
434005
434006
434007
434008
434009
434010
434011
434012
434013
434014
434015
434016
434017
434018
434019
434020
434021
434022
434023
434024
434025
434026
434027
434028
434029
434030
434031
434032
434033
434034
434035
434036
434037
434038
434039
434040
434041
434042
434043
434044
434045
434046
434047
434048
434049
434050
434051
434052
434053
434054
434055
434056
434057
434058
434059
434060
434061
434062
434063
434064
434065
434066
434067
434068
434069
434070
434071
434072
434073
434074
434075
434076
434077
434078
434079
434080
434081
434082
434083
434084
434085
434086
434087
434088
434089
434090
434091
434092
434093
434094
434095
434096
434097
434098
434099
434100
434101
434102
434103
434104
434105
434106
434107
434108
434109
434110
434111
434112
434113
434114
434115
434116
434117
434118
434119
434120
434121
434122
434123
434124
434125
434126
43412

In [ ]:
dim_match_squad

,Match_Squad_Key,Match,Team,Player_Version,Sub
0,1,8479,97,233074.0,No
1,2,8479,97,233111.0,No
2,3,8479,97,233150.0,No
3,4,8479,97,233110.0,No
4,5,8479,97,233096.0,No
...,...,...,...,...,...
438988,438989,12622,136,51506.0,Yes
438989,438990,12622,136,51352.0,Yes
438990,438991,12622,136,51385.0,Yes
438991,438992,12622,136,51400.0,Yes


In [ ]:
dim_match_squad.isna().sum()

Match_Squad_Key         0
Match                   0
Team                    0
Player_Version     150831
Sub                     0
dtype: int64

<h3>8. Dim MatchGoals</h3>

In [ ]:
fbref_matchgoals_new = fbref_matchgoals

In [ ]:
fbref_matchgoals_new

,Match_Id,Team,Minute,Player_Name,Type_Of_Goal,Is_Home_Team
0,39b738ac,Saint-Étienne,10,Unknown,Unknown,Yes
1,39b738ac,Strasbourg,32,Unknown,Unknown,No
2,39b738ac,Saint-Étienne,41,Hernani,Normal,Yes
3,39b738ac,Saint-Étienne,43,Unknown,Unknown,Yes
4,39b738ac,Strasbourg,44,Jean-Eudes Aholou,Normal,No
...,...,...,...,...,...,...
182193,f6279636,Empoli,77,Unknown,Unknown,No
182194,f6279636,Lazio,85,Unknown,Unknown,Yes
182195,f6279636,Lazio,88,Unknown,Unknown,Yes
182196,f6279636,Lazio,89,Matías Vecino,Normal,Yes


In [ ]:
fbref_matchgoals_new.count()

Match_Id        182198
Team            182198
Minute          182198
Player_Name     182198
Type_Of_Goal    182198
Is_Home_Team    182198
dtype: int64

In [ ]:
different_team = set(dim_team['Team_Name']) - set(fbref_matchgoals_new['Team'])
different_team

set()

In [ ]:
dim_match_goals = pd.DataFrame({'Match_Goal_Key': [],
                                'Match': [],
                                'Team': [],
                                'Player_Name': [],
                                'Minute': [],
                                'Type_Of_Goal': []})

list_match_goals_key = []
list_match = []
list_team = []
#list_player_name = []

for i in range(0,len(fbref_matchgoals_new)):
    print(i)
    list_match_goals_key.append(i+1)

    temp_df = dim_match[dim_match['Match_Id'] == fbref_matchgoals_new['Match_Id'].iloc[i]]
    if (len(temp_df) > 0):
        list_match.append(temp_df['Match_Key'].iloc[0])
    else:
        list_match.append(np.nan)

    temp_df = dim_team[dim_team['Team_Name'] == fbref_matchgoals_new['Team'].iloc[i]]
    if (len(temp_df) > 0):
        list_team.append(temp_df['Team_Key'].iloc[0])
    else:
        list_team.append(np.nan)

    #sofifa_df = merge[(merge['Match_Id'] == fbref_matchgoals_new['Match_Id'].iloc[i]) &
    #                  (merge['Team'] == fbref_matchgoals_new['Team'].iloc[i]) &
    #                  (merge['Player_Name'] == fbref_matchgoals_new['Player_Name'].iloc[i])]

    #if (len(sofifa_df) > 0):
    #    sofifa_id = sofifa_df['Sofifa_Id'].iloc[0]
    #else:
    #    sofifa_id = np.nan

    #if (~np.isnan(sofifa_id)):

    #    temp_df = dim_player[dim_player['Player_Id'] == sofifa_id]
    #    if (len(temp_df) > 0):
    #        list_player_name.append(temp_df['Player_Key'].iloc[0])
    #    else:
    #        list_player_name.append(np.nan)
    #else:
    #    list_player.append(np.nan)

dim_match_goals['Match_Goal_Key'] = list_match_goals_key
dim_match_goals['Match'] = list_match
dim_match_goals['Team'] = list_team
dim_match_goals['Player_Name'] = fbref_matchgoals_new['Player_Name'].to_list()
dim_match_goals['Minute'] = fbref_matchgoals_new['Minute'].to_list()
dim_match_goals['Type_Of_Goal'] = fbref_matchgoals_new['Type_Of_Goal'].to_list()


Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
177198
177199
177200
177201
177202
177203
177204
177205
177206
177207
177208
177209
177210
177211
177212
177213
177214
177215
177216
177217
177218
177219
177220
177221
177222
177223
177224
177225
177226
177227
177228
177229
177230
177231
177232
177233
177234
177235
177236
177237
177238
177239
177240
177241
177242
177243
177244
177245
177246
177247
177248
177249
177250
177251
177252
177253
177254
177255
177256
177257
177258
177259
177260
177261
177262
177263
177264
177265
177266
177267
177268
177269
177270
177271
177272
177273
177274
177275
177276
177277
177278
177279
177280
177281
177282
177283
177284
177285
177286
177287
177288
177289
177290
177291
177292
177293
177294
177295
177296
177297
177298
177299
177300
177301
177302
177303
177304
177305
177306
177307
177308
177309
177310
177311
177312
177313
177314
177315
177316
177317
177318
177319
177320
177321
177322
177323
177324
177325
177326
177327
177328
177329
177330
177331
17733

In [ ]:
dim_match_goals

,Match_Goal_Key,Match,Team,Player_Name,Minute,Type_Of_Goal
0,1,51,4,Unknown,10,Unknown
1,2,51,19,Unknown,32,Unknown
2,3,51,4,Hernani,41,Normal
3,4,51,4,Unknown,43,Unknown
4,5,51,19,Jean-Eudes Aholou,44,Normal
...,...,...,...,...,...,...
182193,182194,12622,136,Unknown,77,Unknown
182194,182195,12622,124,Unknown,85,Unknown
182195,182196,12622,124,Unknown,88,Unknown
182196,182197,12622,124,Matías Vecino,89,Normal


In [ ]:
dim_match_goals = dim_match_goals[dim_match_goals['Type_Of_Goal'] != 'Unknown']

In [ ]:
dim_match_goals

,Match_Goal_Key,Match,Team,Player_Name,Minute,Type_Of_Goal
2,3,51,4,Hernani,41,Normal
4,5,51,19,Jean-Eudes Aholou,44,Normal
5,6,51,4,Kévin Monnet-Paquet,56,Normal
7,8,51,19,Jonas Martin,63,Penalty
16,17,287,8,Luiz Araújo,13,Normal
...,...,...,...,...,...,...
182165,182166,12621,131,Rafael Leão,83,Normal
182167,182168,12621,131,Christian Pulisic,86,Normal
182169,182170,12633,138,Walid Cheddira,9,Normal
182182,182183,12622,124,Patric,45+3,Normal


In [ ]:
dim_match_goals.isna().sum()

Match_Goal_Key    0
Match             0
Team              0
Player_Name       0
Minute            0
Type_Of_Goal      0
dtype: int64

<h2>II. Fact Tables</h2>

<h3>1. Fact Player Attributes</h3>

In [ ]:
#sofifa
sofifa_players = pd.read_csv('gs://football_data_etl/football_data_extracted/output-basic/sofifa_players_attr.csv/part-00000-0474179e-867f-4f31-9d30-40c396f830f5-c000.csv',
                                  on_bad_lines="skip")

sofifa_players['Birthday'] = pd.to_datetime(sofifa_players['Birthday'])
sofifa_players['Birthday'] = sofifa_players['Birthday'].dt.strftime('%Y-%m-%d')
sofifa_players['Club_Joined'] = pd.to_datetime(sofifa_players['Club_Joined'])
sofifa_players['Club_Joined'] = sofifa_players['Club_Joined'].dt.strftime('%Y-%m-%d')
sofifa_players['Update_Date'] = pd.to_datetime(sofifa_players['Update_Date'])
sofifa_players['Update_Date'] = sofifa_players['Update_Date'].dt.strftime('%Y-%m-%d')

<ipython-input-78-9b6878256552>:2: DtypeWarning: Columns (45) have mixed types. Specify dtype option on import or set low_memory=False.
  sofifa_players = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/fbref/sofifa_players_attr.csv/part-00000-0474179e-867f-4f31-9d30-40c396f830f5-c000.csv',


In [ ]:
fact_player_attributes = pd.DataFrame({'Player_Version': [],
                                'Acceleration': [], 'Aggression': [], 'Agility': [],
                                'Balance': [], 'Ball_Control': [], 'Composure': [],
                                'Crossing': [], 'Curve': [], 'Dribbling': [],
                                'Finishing': [], 'Fk_Accuracy': [], 'Gk_Diving': [],
                                'Gk_Handling': [], 'Gk_Kicking': [], 'Gk_Positioning': [],
                                'Gk_Reflexes': [], 'Heading_Accuracy': [], 'Height': [],
                                'Interceptions': [], 'Jumping': [], 'Long_Passing': [],
                                'Long_Shots': [], 'Overall_Rating': [], 'Penalties': [],
                                'Positioning': [], 'Potential': [], 'Reactions': [],
                                'Reputation': [], 'Short_Passing': [], 'Shot_Power': [],
                                'Skill_Moves': [], 'Sliding_Tackle': [], 'Sprint_Speed': [],
                                'Stamina': [], 'Standing_Tackle': [], 'Strength': [], 'Value': [],
                                'Vision': [], 'Volleys': [], 'Wage': [],
                                'Weak_Foot': [], 'Weight': []})

list_player_version = []


for i in range(0,len(sofifa_players)):

    print(i)

    player_df = dim_player[dim_player['Player_Id'] == sofifa_players['Sofifa_Id'].iloc[i]]

    if (len(player_df) > 0):
        player_key = player_df['Player_Key'].iloc[0]
    else:
        player_key = np.nan

    temp_df = dim_player_version[dim_player_version['Player'] == player_key]
    if (len(temp_df) > 0):
        list_player_version.append(temp_df['Player_Version_Key'].iloc[0])
    else:
        list_player_version.append(np.nan)


Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
286047
286048
286049
286050
286051
286052
286053
286054
286055
286056
286057
286058
286059
286060
286061
286062
286063
286064
286065
286066
286067
286068
286069
286070
286071
286072
286073
286074
286075
286076
286077
286078
286079
286080
286081
286082
286083
286084
286085
286086
286087
286088
286089
286090
286091
286092
286093
286094
286095
286096
286097
286098
286099
286100
286101
286102
286103
286104
286105
286106
286107
286108
286109
286110
286111
286112
286113
286114
286115
286116
286117
286118
286119
286120
286121
286122
286123
286124
286125
286126
286127
286128
286129
286130
286131
286132
286133
286134
286135
286136
286137
286138
286139
286140
286141
286142
286143
286144
286145
286146
286147
286148
286149
286150
286151
286152
286153
286154
286155
286156
286157
286158
286159
286160
286161
286162
286163
286164
286165
286166
286167
286168
286169
286170
286171
286172
286173
286174
286175
286176
286177
286178
286179
286180
28618

In [ ]:
fact_player_attributes['Player_Version'] = list_player_version
fact_player_attributes['Acceleration'] = sofifa_players['Acceleration']
fact_player_attributes['Aggression'] = sofifa_players['Aggression']
fact_player_attributes['Agility'] = sofifa_players['Agility']
fact_player_attributes['Balance'] = sofifa_players['Balance']
fact_player_attributes['Ball_Control'] = sofifa_players['Ball_Control']
fact_player_attributes['Composure'] = sofifa_players['Composure']
fact_player_attributes['Crossing'] = sofifa_players['Crossing']
fact_player_attributes['Curve'] = sofifa_players['Curve']
fact_player_attributes['Dribbling'] = sofifa_players['Dribbling']
fact_player_attributes['Finishing'] = sofifa_players['Finishing']
fact_player_attributes['Fk_Accuracy'] = sofifa_players['Fk_Accuracy']
fact_player_attributes['Gk_Diving'] = sofifa_players['Gk_Diving']
fact_player_attributes['Gk_Handling'] = sofifa_players['Gk_Handling']
fact_player_attributes['Gk_Kicking'] = sofifa_players['Gk_Kicking']
fact_player_attributes['Gk_Positioning'] = sofifa_players['Gk_Positioning']
fact_player_attributes['Gk_Reflexes'] = sofifa_players['Gk_Reflexes']

fact_player_attributes['Heading_Accuracy'] = sofifa_players['Heading_Accuracy']
fact_player_attributes['Height'] = sofifa_players['Height']
fact_player_attributes['Interceptions'] = sofifa_players['Interceptions']
fact_player_attributes['Jumping'] = sofifa_players['Jumping']
fact_player_attributes['Long_Passing'] = sofifa_players['Long_Passing']
fact_player_attributes['Long_Shots'] = sofifa_players['Long_Shots']
fact_player_attributes['Overall_Rating'] = sofifa_players['Overall_Rating']
fact_player_attributes['Penalties'] = sofifa_players['Penalties']
fact_player_attributes['Positioning'] = sofifa_players['Positioning']
fact_player_attributes['Potential'] = sofifa_players['Potential']
fact_player_attributes['Reactions'] = sofifa_players['Reactions']
fact_player_attributes['Reputation'] = sofifa_players['Reputation']
fact_player_attributes['Short_Passing'] = sofifa_players['Short_Passing']
fact_player_attributes['Shot_Power'] = sofifa_players['Shot_Power']

fact_player_attributes['Skill_Moves'] = sofifa_players['Skill_Moves']
fact_player_attributes['Sliding_Tackle'] = sofifa_players['Sliding_Tackle']
fact_player_attributes['Sprint_Speed'] = sofifa_players['Sprint_Speed']
fact_player_attributes['Stamina'] = sofifa_players['Stamina']
fact_player_attributes['Standing_Tackle'] = sofifa_players['Standing_Tackle']
fact_player_attributes['Strength'] = sofifa_players['Strength']
fact_player_attributes['Value'] = sofifa_players['Value']
fact_player_attributes['Vision'] = sofifa_players['Vision']
fact_player_attributes['Volleys'] = sofifa_players['Volleys']
fact_player_attributes['Wage'] = sofifa_players['Wage']
fact_player_attributes['Weak_Foot'] = sofifa_players['Weak_Foot']
fact_player_attributes['Weight'] = sofifa_players['Weight']

<h3>2. Fact Match Statistics</h3>

Thêm thông số Attack, Midfield, Defense dựa vào Player_Position và Player_Overall_Rating cho mỗi match

In [29]:
match_dataset_model = pd.read_csv("gs://football_data_etl/football_data_extracted/output-basic/match_dataset_model.csv",
                                  encoding='latin1')

In [30]:
match_dataset_model['Home_Team'].unique()

array(['Saint-\x90tienne', 'Lille', 'Nice', 'Guingamp', 'Caen', 'Nantes',
       'Strasbourg', 'Bordeaux', 'Lyon', 'Toulouse', 'Paris S-G', 'Metz',
       'Monaco', 'Montpellier', 'Troyes', 'Rennes', 'Angers', 'Marseille',
       'Dijon', 'Amiens', 'N\x8cmes', 'Reims', 'Brest', 'Lorient', 'Lens',
       'Clermont Foot', 'Auxerre', 'Ajaccio', 'Le Havre', 'West Ham',
       'Everton', 'Chelsea', 'Southampton', 'Crystal Palace',
       'Cardiff City', 'Liverpool', 'Huddersfield', 'Fulham', 'Burnley',
       'Brighton', 'Manchester City', 'Arsenal', 'Tottenham', 'Wolves',
       'Leicester City', 'Bournemouth', 'Newcastle Utd', 'Watford',
       'Manchester Utd', 'Norwich City', 'Aston Villa', 'Sheffield Utd',
       'Leeds United', 'West Brom', 'Brentford', "Nott'ham Forest",
       'Luton Town', 'Swansea City', 'Stoke City', 'La Coru¤a',
       'Athletic Club', 'Villarreal', 'Real Sociedad', 'Valencia',
       'Las Palmas', 'Atl\x82tico Madrid', 'Barcelona', 'Eibar',
       'Real Madrid'

In [31]:
match_dataset_model.loc[match_dataset_model['Home_Team'] == 'Saint-\x90tienne', 'Home_Team'] = 'Saint-Étienne'
match_dataset_model.loc[match_dataset_model['Away_Team'] == 'Saint-\x90tienne', 'Away_Team'] = 'Saint-Étienne'

match_dataset_model.loc[match_dataset_model['Home_Team'] == 'N\x8cmes', 'Home_Team'] = 'Nîmes'
match_dataset_model.loc[match_dataset_model['Away_Team'] == 'N\x8cmes', 'Away_Team'] = 'Nîmes'

match_dataset_model.loc[match_dataset_model['Home_Team'] == 'La Coru¤a', 'Home_Team'] = 'La Coruña'
match_dataset_model.loc[match_dataset_model['Away_Team'] == 'La Coru¤a', 'Away_Team'] = 'La Coruña'

match_dataset_model.loc[match_dataset_model['Home_Team'] == 'Atl\x82tico Madrid', 'Home_Team'] = 'Atlético Madrid'
match_dataset_model.loc[match_dataset_model['Away_Team'] == 'Atl\x82tico Madrid', 'Away_Team'] = 'Atlético Madrid'

match_dataset_model.loc[match_dataset_model['Home_Team'] == 'M\xa0laga', 'Home_Team'] = 'Málaga'
match_dataset_model.loc[match_dataset_model['Away_Team'] == 'M\xa0laga', 'Away_Team'] = 'Málaga'

match_dataset_model.loc[match_dataset_model['Home_Team'] == 'Almer¡a', 'Home_Team'] = 'Almería'
match_dataset_model.loc[match_dataset_model['Away_Team'] == 'Almer¡a', 'Away_Team'] = 'Almería'

match_dataset_model.loc[match_dataset_model['Home_Team'] == 'Alav\x82s', 'Home_Team'] = 'Alavés'
match_dataset_model.loc[match_dataset_model['Away_Team'] == 'Alav\x82s', 'Away_Team'] = 'Alavés'

match_dataset_model.loc[match_dataset_model['Home_Team'] == 'Legan\x82s', 'Home_Team'] = 'Leganés'
match_dataset_model.loc[match_dataset_model['Away_Team'] == 'Legan\x82s', 'Away_Team'] = 'Leganés'

match_dataset_model.loc[match_dataset_model['Home_Team'] == 'C\xa0diz', 'Home_Team'] = 'Cádiz'
match_dataset_model.loc[match_dataset_model['Away_Team'] == 'C\xa0diz', 'Away_Team'] = 'Cádiz'

match_dataset_model.loc[match_dataset_model['Home_Team'] == 'K\x94ln', 'Home_Team'] = 'Köln'
match_dataset_model.loc[match_dataset_model['Away_Team'] == 'K\x94ln', 'Away_Team'] = 'Köln'

match_dataset_model.loc[match_dataset_model['Home_Team'] == 'D\x81sseldorf', 'Home_Team'] = 'Düsseldorf'
match_dataset_model.loc[match_dataset_model['Away_Team'] == 'D\x81sseldorf', 'Away_Team'] = 'Düsseldorf'

match_dataset_model.loc[match_dataset_model['Home_Team'] == 'N\x81rnberg', 'Home_Team'] = 'Nürnberg'
match_dataset_model.loc[match_dataset_model['Away_Team'] == 'N\x81rnberg', 'Away_Team'] = 'Nürnberg'

match_dataset_model.loc[match_dataset_model['Home_Team'] == 'Greuther F\x81rth', 'Home_Team'] = 'Greuther Fürth'
match_dataset_model.loc[match_dataset_model['Away_Team'] == 'Greuther F\x81rth', 'Away_Team'] = 'Greuther Fürth'

In [32]:
match_dataset_model

,Match_Id,Match_Date,Home_Team,Home_Manager,Home_Captain,Home_Formation,Home_Possession,Home_Fouls,Home_Corners,Home_Crosses,...,Away_Cmp_Passes,Away_Att_Passes,Away_Cmp_percent_Passes,Away_PrgP_Passes,Away_Carries_Carries,Away_PrgC_Carries,Away_Att_Take_Ons,Away_Succ_Take_Ons,Home_Score,Away_Score
0,39b738ac,11/24/2017,Saint-Étienne,Unknown,Unknown,4-2-3-1,59,14,7,33,...,293,410,71.5,23,297,18,12,9,2,2
1,ed0f2e13,4/28/2018,Lille,Unknown,Unknown,4-2-3-1,50,11,3,12,...,378,511,74.0,38,338,15,23,12,3,1
2,de5b03fe,2/3/2018,Nice,Unknown,Unknown,4/3/2003,75,14,11,34,...,204,279,73.1,26,246,17,21,13,0,1
3,43ebed63,12/20/2017,Guingamp,Unknown,Unknown,4-2-3-1,80,13,10,47,...,90,179,50.3,11,73,5,7,3,2,1
4,1aef5397,8/12/2017,Caen,Unknown,Unknown,4/3/2003,47,12,8,32,...,340,454,74.9,32,260,16,16,10,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12599,9492fa68,5/25/2024,Milan,Unknown,Unknown,4-2-3-1,66,6,13,27,...,234,300,78.0,24,158,17,9,2,3,3
12600,d63c561d,5/18/2024,Lecce,Unknown,Unknown,4/4/2002,44,7,2,15,...,509,593,85.8,50,379,19,13,7,0,2
12601,dec7c7ae,5/11/2024,Milan,Unknown,Unknown,4-2-3-1,66,11,6,15,...,275,350,78.6,21,230,15,17,6,5,1
12602,ab8f65b4,5/19/2024,Monza,Unknown,Unknown,4-2-3-1,70,14,3,23,...,197,278,70.9,25,152,20,17,9,0,1


In [33]:
match_dataset_model = pd.merge(match_dataset_model, fbref_matchinfos[['Match_Id','Season']], on="Match_Id")

In [34]:
match_dataset_model = match_dataset_model[match_dataset_model['Season'] != 2018]
match_dataset_model = match_dataset_model.drop('Season', axis=1)

Xóa các Match với không tìm thấy Sofifa_Id nào của một Team

In [35]:
sofifa_id_counts = merge.groupby(['Match_Id', 'Team'])['Sofifa_Id'].count()
sofifa_id_counts

Match_Id  Team           
00023357  Hoffenheim          2
          Leverkusen          4
0006415c  Atalanta           19
          Udinese            23
0012462a  Atlético Madrid    21
                             ..
ffeee70d  Nantes             18
fffbcb62  Monaco             18
          Saint-Étienne      17
fffdfc41  Bochum             16
          Eint Frankfurt     14
Name: Sofifa_Id, Length: 21562, dtype: int64

In [36]:
sofifa_id_counts = sofifa_id_counts.reset_index()

In [37]:
ids_team_with_zero_count = sofifa_id_counts[(sofifa_id_counts['Sofifa_Id'] == 0)][['Match_Id', 'Team']]

In [38]:
merge_filtered = pd.merge(merge, ids_team_with_zero_count, on=['Match_Id', 'Team'], how='outer', indicator=True)
merge_filtered = merge_filtered[merge_filtered['_merge'] != 'both']

In [39]:
len(merge)

438993

In [40]:
len(merge_filtered)

404471

In [41]:
len(merge['Match_Id'].unique())

10781

In [42]:
len(merge_filtered['Match_Id'].unique())

10213

In [43]:
merge_filtered['Team'].unique()

array(["M'Gladbach", 'Eint Frankfurt', 'Hertha BSC', 'Wolfsburg',
       'Leverkusen', 'Dortmund', 'Werder Bremen', 'Freiburg',
       'Bayern Munich', 'Hoffenheim', 'Hannover 96', 'Stuttgart',
       'Schalke 04', 'Nürnberg', 'RB Leipzig', 'Mainz 05', 'Augsburg',
       'Köln', 'Union Berlin', 'Bochum', 'Heidenheim', 'Darmstadt 98',
       'Atlético Madrid', 'Getafe', 'Villarreal', 'Real Sociedad',
       'Valladolid', 'Real Madrid', 'Betis', 'Alavés', 'Girona',
       'Levante', 'Leganés', 'Athletic Club', 'Espanyol', 'Eibar',
       'Barcelona', 'Sevilla', 'Celta Vigo', 'Valencia', 'Huesca',
       'Rayo Vallecano', 'Granada', 'Osasuna', 'Mallorca', 'Elche',
       'Cádiz', 'Almería', 'Las Palmas', 'Nîmes', 'Nantes', 'Dijon',
       'Marseille', 'Saint-Étienne', 'Bordeaux', 'Strasbourg', 'Caen',
       'Lille', 'Paris S-G', 'Toulouse', 'Lyon', 'Montpellier', 'Angers',
       'Amiens', 'Monaco', 'Nice', 'Rennes', 'Guingamp', 'Reims', 'Metz',
       'Brest', 'Lorient', 'Lens', 'Troyes

In [44]:
merge_filtered[~merge_filtered['Team'].isin(match_dataset_model['Home_Team'])]['Team']

Series([], Name: Team, dtype: object)

Qui định các các Position ứng với 3 Lines: Attack, Midfield, Defense

In [45]:
attack_line = ["LS", "ST", "RS","LW", "LF", "CF", "RF", "RW"]
midfield_line = ["LAM", "CAM", "RAM","LM", "LCM", "CM", "RCM", "RM","LDM", "CDM", "RDM"]
defense_line = ["LB", "LCB", "CB", "RCB", "RB","LWB","RWB"]

In [46]:
new_attrs = pd.DataFrame({'Match_Id': [],
                        'Home_Attack': [],
                        'Home_Midfield': [],
                        'Home_Defense': [],
                        'Away_Attack': [],
                        'Away_Midfield': [],
                        'Away_Defense': []})
atr_match_id = []
atr_home_attack = []
atr_home_midfield = []
atr_home_defense = []
atr_away_attack = []
atr_away_midfield = []
atr_away_defense = []

In [47]:
for i in range(0, len(match_dataset_model)):

    match_id = match_dataset_model.iloc[i, match_dataset_model.columns.get_loc('Match_Id')]

    home_team = match_dataset_model.iloc[i, match_dataset_model.columns.get_loc('Home_Team')]
    away_team = match_dataset_model.iloc[i, match_dataset_model.columns.get_loc('Away_Team')]

    #
    current_match_df = merge_filtered[merge_filtered['Match_Id'] == match_id]

    print(i, match_id, home_team, away_team, len(current_match_df))

    if (len(current_match_df) > 0):

      list_home_attack = []
      list_home_midfield = []
      list_home_defense = []
      list_away_attack = []
      list_away_midfield = []
      list_away_defense = []

      for player in current_match_df.itertuples(index=False):

          if not np.isnan(player.Sofifa_Id):
              player_positions = player.Player_Position.split(' ')
              overall_rating = player.Player_Overall_Rating

          #home_team
          if (player.Team == home_team) and not np.isnan(player.Sofifa_Id):

              for position in player_positions:
                  if (position in attack_line):
                      list_home_attack.append(overall_rating)
                  if (position in midfield_line):
                      list_home_midfield.append(overall_rating)
                  if (position in defense_line):
                      list_home_defense.append(overall_rating)
          #away_team
          if (player.Team == away_team) and not np.isnan(player.Sofifa_Id):

              for position in player_positions:
                  if (position in attack_line):
                      list_away_attack.append(overall_rating)
                  if (position in midfield_line):
                      list_away_midfield.append(overall_rating)
                  if (position in defense_line):
                      list_away_defense.append(overall_rating)

      atr_match_id.append(match_id)

      if (len(list_home_attack) > 0):
        atr_home_attack.append(sum(list_home_attack) / len(list_home_attack))
      else:
        atr_home_attack.append(0)
      if (len(list_home_midfield) > 0):
        atr_home_midfield.append(sum(list_home_midfield) / len(list_home_midfield))
      else:
        atr_home_midfield.append(0)
      if (len(list_home_defense) > 0):
        atr_home_defense.append(sum(list_home_defense) / len(list_home_defense))
      else:
        atr_home_defense.append(0)

      if (len(list_away_attack) > 0):
        atr_away_attack.append(sum(list_away_attack) / len(list_away_attack))
      else:
        atr_away_attack.append(0)
      if (len(list_away_midfield) > 0):
        atr_away_midfield.append(sum(list_away_midfield) / len(list_away_midfield))
      else:
        atr_away_midfield.append(0)
      if (len(list_away_defense) > 0):
        atr_away_defense.append(sum(list_away_defense) / len(list_away_defense))
      else:
        atr_away_defense.append(0)


    else:
      atr_match_id.append(match_id)
      atr_home_attack.append(0)
      atr_home_midfield.append(0)
      atr_home_defense.append(0)
      atr_away_attack.append(0)
      atr_away_midfield.append(0)
      atr_away_defense.append(0)

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
5683 5bbb6e37 Alavés Real Sociedad 42
5684 99b54d59 Villarreal Sevilla 44
5685 20c84d6e Atlético Madrid Getafe 41
5686 368c0ef5 Valencia Barcelona 44
5687 20c47249 Real Sociedad Getafe 45
5688 a7352bb8 Granada Valencia 41
5689 170e5383 Sevilla Osasuna 46
5690 f8700700 Athletic Club Mallorca 22
5691 3334afc1 Valencia Alavés 44
5692 140f4110 Valencia Real Madrid 44
5693 6d1df89a Elche Celta Vigo 45
5694 2b813dce Granada Betis 46
5695 bc4e66b1 Elche Betis 46
5696 ed5b7ecb Sevilla Real Sociedad 45
5697 59a8ec39 Celta Vigo Barcelona 42
5698 43ebabcc Elche Espanyol 46
5699 b3946deb Espanyol Cádiz 46
5700 ce97f705 Villarreal Espanyol 45
5701 777c8d55 Athletic Club Real Sociedad 43
5702 557b4f21 Osasuna Atlético Madrid 42
5703 cdc92553 Villarreal Osasuna 46
5704 43a5cb2e Villarreal Cádiz 46
5705 8bb438aa Athletic Club Osasuna 44
5706 e4557ea7 Real Madrid Cádiz 44
5707 ca6be7cc Levante Espanyol 46
5708 ac93ead3 Atlético Madrid Villarreal 

In [48]:
new_attrs['Match_Id'] = atr_match_id
new_attrs['Home_Attack'] = atr_home_attack
new_attrs['Home_Midfield'] = atr_home_midfield
new_attrs['Home_Defense'] = atr_home_defense
new_attrs['Away_Attack'] = atr_away_attack
new_attrs['Away_Midfield'] = atr_away_midfield
new_attrs['Away_Defense'] = atr_away_defense

In [49]:
match_dataset_model_new = pd.merge(match_dataset_model, new_attrs, on='Match_Id', how='inner')
match_dataset_model_new

,Match_Id,Match_Date,Home_Team,Home_Manager,Home_Captain,Home_Formation,Home_Possession,Home_Fouls,Home_Corners,Home_Crosses,...,Away_Att_Take_Ons,Away_Succ_Take_Ons,Home_Score,Away_Score,Home_Attack,Home_Midfield,Home_Defense,Away_Attack,Away_Midfield,Away_Defense
0,4eccf13c,1/16/2019,Nîmes,Unknown,Unknown,4/4/2002,57,13,5,19,...,8,7,1,0,70.166667,70.583333,68.750000,74.333333,74.187500,73.571429
1,da4a3dd0,2/8/2019,Dijon,Unknown,Unknown,4/3/2003,41,11,3,17,...,15,10,1,2,73.800000,71.857143,70.333333,76.714286,77.916667,74.875000
2,6a8b98ba,4/14/2019,Saint-Étienne,Unknown,Unknown,3/4/2003,50,13,8,24,...,12,10,3,0,71.500000,76.000000,71.857143,72.600000,68.230769,70.333333
3,25268197,5/24/2019,Nantes,Unknown,Unknown,4/3/2003,51,7,4,20,...,13,10,0,1,69.625000,72.928571,70.333333,72.200000,71.769231,73.000000
4,2bfe33f3,1/11/2019,Caen,Unknown,Unknown,4-2-3-1,56,14,4,25,...,11,4,1,3,72.400000,73.285714,69.272727,75.600000,74.800000,71.700000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10678,9492fa68,5/25/2024,Milan,Unknown,Unknown,4-2-3-1,66,6,13,27,...,9,2,3,3,81.375000,79.285714,77.800000,70.500000,70.210526,69.625000
10679,d63c561d,5/18/2024,Lecce,Unknown,Unknown,4/4/2002,44,7,2,15,...,13,7,0,2,69.181818,67.307692,71.125000,78.250000,75.500000,73.307692
10680,dec7c7ae,5/11/2024,Milan,Unknown,Unknown,4-2-3-1,66,11,6,15,...,17,6,5,1,81.222222,77.428571,77.571429,69.888889,71.444444,70.600000
10681,ab8f65b4,5/19/2024,Monza,Unknown,Unknown,4-2-3-1,70,14,3,23,...,17,9,0,1,71.545455,73.125000,73.636364,68.727273,69.111111,68.416667


In [50]:
match_dataset_model_new = match_dataset_model_new.loc[:,['Match_Id', 'Match_Date',
                                'Home_Team', 'Home_Manager', 'Home_Captain',
                                'Home_Formation', 'Home_Possession', 'Home_Fouls', 'Home_Corners', 'Home_Crosses',
                                'Home_Aerials_Won', 'Home_Clearances', 'Home_Offsides', 'Home_Goal_Kicks',
                                'Home_Throw_Ins', 'Home_Long_Balls', 'Home_Total_Players_Stats', 'Home_Minutes',
                                'Home_Gls', 'Home_Ast', 'Home_PK', 'Home_PK_Att',
                                'Home_Sh','Home_SoT', 'Home_CrdY', 'Home_CrdR',
                                'Home_Touches','Home_Tkl','Home_Int','Home_Blocks',
                                'Home_xG','Home_npxG', 'Home_xAG',
                                'Home_SCA', 'Home_GCA',
                                'Home_Cmp_Passes', 'Home_Att_Passes', 'Home_Cmp_percent_Passes', 'Home_PrgP_Passes',
                                'Home_Carries_Carries', 'Home_PrgC_Carries',
                                'Home_Att_Take_Ons', 'Home_Succ_Take_Ons',
                                'Away_Team', 'Away_Manager', 'Away_Captain',
                                'Away_Formation', 'Away_Possession', 'Away_Fouls', 'Away_Corners', 'Away_Crosses',
                                'Away_Aerials_Won', 'Away_Clearances', 'Away_Offsides', 'Away_Goal_Kicks',
                                'Away_Throw_Ins', 'Away_Long_Balls', 'Away_Total_Players_Stats', 'Away_Minutes',
                                'Away_Gls', 'Away_Ast', 'Away_PK', 'Away_PK_Att',
                                'Away_Sh','Away_SoT', 'Away_CrdY', 'Away_CrdR',
                                'Away_Touches','Away_Tkl','Away_Int','Away_Blocks',
                                'Away_xG','Away_npxG', 'Away_xAG',
                                'Away_SCA', 'Away_GCA',
                                'Away_Cmp_Passes', 'Away_Att_Passes', 'Away_Cmp_percent_Passes', 'Away_PrgP_Passes',
                                'Away_Carries_Carries', 'Away_PrgC_Carries',
                                'Away_Att_Take_Ons', 'Away_Succ_Take_Ons',
                                'Home_Attack', 'Home_Midfield', 'Home_Defense',
                                'Away_Attack', 'Away_Midfield', 'Away_Defense',
                                'Home_Score','Away_Score']]
match_dataset_model_new

,Match_Id,Match_Date,Home_Team,Home_Manager,Home_Captain,Home_Formation,Home_Possession,Home_Fouls,Home_Corners,Home_Crosses,...,Away_Att_Take_Ons,Away_Succ_Take_Ons,Home_Attack,Home_Midfield,Home_Defense,Away_Attack,Away_Midfield,Away_Defense,Home_Score,Away_Score
0,4eccf13c,1/16/2019,Nîmes,Unknown,Unknown,4/4/2002,57,13,5,19,...,8,7,70.166667,70.583333,68.750000,74.333333,74.187500,73.571429,1,0
1,da4a3dd0,2/8/2019,Dijon,Unknown,Unknown,4/3/2003,41,11,3,17,...,15,10,73.800000,71.857143,70.333333,76.714286,77.916667,74.875000,1,2
2,6a8b98ba,4/14/2019,Saint-Étienne,Unknown,Unknown,3/4/2003,50,13,8,24,...,12,10,71.500000,76.000000,71.857143,72.600000,68.230769,70.333333,3,0
3,25268197,5/24/2019,Nantes,Unknown,Unknown,4/3/2003,51,7,4,20,...,13,10,69.625000,72.928571,70.333333,72.200000,71.769231,73.000000,0,1
4,2bfe33f3,1/11/2019,Caen,Unknown,Unknown,4-2-3-1,56,14,4,25,...,11,4,72.400000,73.285714,69.272727,75.600000,74.800000,71.700000,1,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10678,9492fa68,5/25/2024,Milan,Unknown,Unknown,4-2-3-1,66,6,13,27,...,9,2,81.375000,79.285714,77.800000,70.500000,70.210526,69.625000,3,3
10679,d63c561d,5/18/2024,Lecce,Unknown,Unknown,4/4/2002,44,7,2,15,...,13,7,69.181818,67.307692,71.125000,78.250000,75.500000,73.307692,0,2
10680,dec7c7ae,5/11/2024,Milan,Unknown,Unknown,4-2-3-1,66,11,6,15,...,17,6,81.222222,77.428571,77.571429,69.888889,71.444444,70.600000,5,1
10681,ab8f65b4,5/19/2024,Monza,Unknown,Unknown,4-2-3-1,70,14,3,23,...,17,9,71.545455,73.125000,73.636364,68.727273,69.111111,68.416667,0,1


Fact Match Statistics

In [ ]:
#existing files
#match_dataset_model = pd.read_csv("gs://football-data-etl/football-data-extracted/output-basic/match_dataset_model_new.csv")

In [51]:
match_dataset_model = match_dataset_model_new

In [52]:
unique_values1 = match_dataset_model.loc[~match_dataset_model['Home_Team'].isin(dim_team['Team_Name']), 'Home_Team'].unique()
unique_values1

array(['Gladbach'], dtype=object)

In [53]:
match_dataset_model.loc[match_dataset_model['Home_Team'] == 'Gladbach', 'Home_Team'] = "M'Gladbach"
match_dataset_model.loc[match_dataset_model['Away_Team'] == 'Gladbach', 'Away_Team'] = "M'Gladbach"

In [54]:
unique_values2 = match_dataset_model.loc[~match_dataset_model['Home_Team'].isin(dim_team['Team_Name']), 'Home_Team'].unique()
unique_values2

array([], dtype=object)

In [55]:
fact_match_statistics = pd.DataFrame({'Match': [],
                                'Home_Team': [],
                                'Home_Possession': [], 'Home_Fouls': [], 'Home_Corners': [],
                                'Home_Crosses': [], 'Home_Aerials_Won': [], 'Home_Clearances': [],
                                'Home_Offsides': [], 'Home_Goal_Kicks': [], 'Home_Throw_Ins': [],
                                'Home_Long_Balls': [], 'Home_Total_Players_Stats': [], 'Home_Minutes': [],
                                'Home_Gls': [], 'Home_Ast': [], 'Home_PK': [],
                                'Home_PK_Att': [], 'Home_Sh': [], 'Home_SoT': [],
                                'Home_CrdY': [], 'Home_CrdR': [], 'Home_Touches': [],
                                'Home_Tkl': [], 'Home_Int': [], 'Home_Blocks': [],
                                'Home_xG': [], 'Home_npxG': [], 'Home_xAG': [],
                                'Home_SCA': [], 'Home_GCA': [], 'Home_Cmp_Passes': [],
                                'Home_Att_Passes': [], 'Home_Cmp_percent_Passes': [], 'Home_PrgP_Passes': [],
                                'Home_Carries_Carries': [], 'Home_PrgC_Carries': [], 'Home_Att_Take_Ons': [],
                                'Home_Succ_Take_Ons': [],

                                'Away_Team': [],
                                'Away_Possession': [],'Away_Fouls': [],'Away_Corners': [],
                                'Away_Crosses': [],'Away_Aerials_Won': [],'Away_Clearances': [],
                                'Away_Offsides': [],'Away_Goal_Kicks': [],'Away_Throw_Ins': [],
                                'Away_Long_Balls': [],'Away_Total_Players_Stats': [],'Away_Minutes': [],
                                'Away_Gls': [],'Away_Ast': [],'Away_PK': [],
                                'Away_PK_Att': [],'Away_Sh': [],'Away_SoT': [],
                                'Away_CrdY': [],'Away_CrdR': [],'Away_Touches': [],
                                'Away_Tkl': [],'Away_Int': [],'Away_Blocks': [],
                                'Away_xG': [],'Away_npxG': [],'Away_xAG': [],
                                'Away_SCA': [],'Away_GCA': [],'Away_Cmp_Passes': [],
                                'Away_Att_Passes': [],'Away_Cmp_percent_Passes': [],'Away_PrgP_Passes': [],
                                'Away_Carries_Carries': [],'Away_PrgC_Carries': [],'Away_Att_Take_Ons': [],
                                'Away_Succ_Take_Ons': [],

                                'Home_Attack': [],'Home_Midfield': [],'Home_Defense': [],
                                'Away_Attack': [],'Away_Midfield': [],'Away_Defense': [],
                                'Home_Score': [],'Away_Score': []})

list_match = []
list_home_team = []
list_away_team = []

for i in range(0,len(match_dataset_model)):


    temp_df = dim_match[dim_match['Match_Id'] == match_dataset_model['Match_Id'].iloc[i]]
    if (len(temp_df) > 0):
        list_match.append(temp_df['Match_Key'].iloc[0])
    else:
        list_match.append(np.nan)

    temp_df = dim_team[dim_team['Team_Name'] == match_dataset_model['Home_Team'].iloc[i]]
    if (len(temp_df) > 0):

        print(i, match_dataset_model['Home_Team'].iloc[i])

        list_home_team.append(temp_df['Team_Key'].iloc[0])
    else:
        list_home_team.append(np.nan)



    temp_df = dim_team[dim_team['Team_Name'] == match_dataset_model_new['Away_Team'].iloc[i]]
    if (len(temp_df) > 0):
        list_away_team.append(temp_df['Team_Key'].iloc[0])
    else:
        list_away_team.append(np.nan)

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
5683 Alavés
5684 Villarreal
5685 Atlético Madrid
5686 Valencia
5687 Real Sociedad
5688 Granada
5689 Sevilla
5690 Athletic Club
5691 Valencia
5692 Valencia
5693 Elche
5694 Granada
5695 Elche
5696 Sevilla
5697 Celta Vigo
5698 Elche
5699 Espanyol
5700 Villarreal
5701 Athletic Club
5702 Osasuna
5703 Villarreal
5704 Villarreal
5705 Athletic Club
5706 Real Madrid
5707 Levante
5708 Atlético Madrid
5709 Real Sociedad
5710 Barcelona
5711 Barcelona
5712 Celta Vigo
5713 Rayo Vallecano
5714 Granada
5715 Granada
5716 Barcelona
5717 Real Madrid
5718 Betis
5719 Barcelona
5720 Athletic Club
5721 Real Madrid
5722 Mallorca
5723 Getafe
5724 Rayo Vallecano
5725 Real Madrid
5726 Villarreal
5727 Osasuna
5728 Levante
5729 Rayo Vallecano
5730 Rayo Vallecano
5731 Alavés
5732 Getafe
5733 Celta Vigo
5734 Villarreal
5735 Levante
5736 Granada
5737 Atlético Madrid
5738 Espanyol
5739 Espanyol
5740 Granada
5741 Levante
5742 Sevilla
5743 Real Sociedad
5744 Celta

In [56]:
fact_match_statistics['Match'] = list_match
fact_match_statistics['Home_Team'] = list_home_team
fact_match_statistics['Home_Possession'] = match_dataset_model['Home_Possession'].to_list()
fact_match_statistics['Home_Fouls'] = match_dataset_model['Home_Fouls'].to_list()
fact_match_statistics['Home_Corners'] = match_dataset_model['Home_Corners'].to_list()
fact_match_statistics['Home_Crosses'] = match_dataset_model['Home_Crosses'].to_list()
fact_match_statistics['Home_Aerials_Won'] = match_dataset_model['Home_Aerials_Won'].to_list()
fact_match_statistics['Home_Clearances'] = match_dataset_model['Home_Clearances'].to_list()
fact_match_statistics['Home_Offsides'] = match_dataset_model['Home_Offsides'].to_list()
fact_match_statistics['Home_Goal_Kicks'] = match_dataset_model['Home_Goal_Kicks'].to_list()
fact_match_statistics['Home_Throw_Ins'] = match_dataset_model['Home_Throw_Ins'].to_list()
fact_match_statistics['Home_Long_Balls'] = match_dataset_model['Home_Long_Balls'].to_list()
fact_match_statistics['Home_Total_Players_Stats'] = match_dataset_model['Home_Total_Players_Stats'].to_list()
fact_match_statistics['Home_Minutes'] = match_dataset_model['Home_Minutes'].to_list()
fact_match_statistics['Home_Gls'] = match_dataset_model['Home_Gls'].to_list()
fact_match_statistics['Home_Ast'] = match_dataset_model['Home_Ast'].to_list()
fact_match_statistics['Home_PK'] = match_dataset_model['Home_PK'].to_list()
fact_match_statistics['Home_PK_Att'] = match_dataset_model['Home_PK_Att'].to_list()
fact_match_statistics['Home_Sh'] = match_dataset_model['Home_Sh'].to_list()
fact_match_statistics['Home_SoT'] = match_dataset_model['Home_SoT'].to_list()
fact_match_statistics['Home_CrdY'] = match_dataset_model['Home_CrdY'].to_list()
fact_match_statistics['Home_CrdR'] = match_dataset_model['Home_CrdR'].to_list()
fact_match_statistics['Home_Touches'] = match_dataset_model['Home_Touches'].to_list()
fact_match_statistics['Home_Tkl'] = match_dataset_model['Home_Tkl'].to_list()
fact_match_statistics['Home_Int'] = match_dataset_model['Home_Int'].to_list()
fact_match_statistics['Home_Blocks'] = match_dataset_model['Home_Blocks'].to_list()
fact_match_statistics['Home_Touches'] = match_dataset_model['Home_Touches'].to_list()
fact_match_statistics['Home_Tkl'] = match_dataset_model['Home_Tkl'].to_list()
fact_match_statistics['Home_Int'] = match_dataset_model['Home_Int'].to_list()
fact_match_statistics['Home_Blocks'] = match_dataset_model['Home_Blocks'].to_list()
fact_match_statistics['Home_xG'] = match_dataset_model['Home_xG'].to_list()
fact_match_statistics['Home_npxG'] = match_dataset_model['Home_npxG'].to_list()
fact_match_statistics['Home_xAG'] = match_dataset_model['Home_xAG'].to_list()
fact_match_statistics['Home_SCA'] = match_dataset_model['Home_SCA'].to_list()
fact_match_statistics['Home_GCA'] = match_dataset_model['Home_GCA'].to_list()
fact_match_statistics['Home_Cmp_Passes'] = match_dataset_model['Home_Cmp_Passes'].to_list()
fact_match_statistics['Home_Att_Passes'] = match_dataset_model['Home_Att_Passes'].to_list()
fact_match_statistics['Home_Cmp_percent_Passes'] = match_dataset_model['Home_Cmp_percent_Passes'].to_list()
fact_match_statistics['Home_PrgP_Passes'] = match_dataset_model['Home_PrgP_Passes'].to_list()
fact_match_statistics['Home_Carries_Carries'] = match_dataset_model['Home_Carries_Carries'].to_list()
fact_match_statistics['Home_PrgC_Carries'] = match_dataset_model['Home_PrgC_Carries'].to_list()
fact_match_statistics['Home_Att_Take_Ons'] = match_dataset_model['Home_Att_Take_Ons'].to_list()
fact_match_statistics['Home_Succ_Take_Ons'] = match_dataset_model['Home_Succ_Take_Ons'].to_list()

fact_match_statistics['Away_Team'] = list_away_team
fact_match_statistics['Away_Possession'] = match_dataset_model['Away_Possession'].to_list()
fact_match_statistics['Away_Fouls'] = match_dataset_model['Away_Fouls'].to_list()
fact_match_statistics['Away_Corners'] = match_dataset_model['Away_Corners'].to_list()
fact_match_statistics['Away_Crosses'] = match_dataset_model['Away_Crosses'].to_list()
fact_match_statistics['Away_Aerials_Won'] = match_dataset_model['Away_Aerials_Won'].to_list()
fact_match_statistics['Away_Clearances'] = match_dataset_model['Away_Clearances'].to_list()
fact_match_statistics['Away_Offsides'] = match_dataset_model['Away_Offsides'].to_list()
fact_match_statistics['Away_Goal_Kicks'] = match_dataset_model['Away_Goal_Kicks'].to_list()
fact_match_statistics['Away_Throw_Ins'] = match_dataset_model['Away_Throw_Ins'].to_list()
fact_match_statistics['Away_Long_Balls'] = match_dataset_model['Away_Long_Balls'].to_list()
fact_match_statistics['Away_Total_Players_Stats'] = match_dataset_model['Away_Total_Players_Stats'].to_list()
fact_match_statistics['Away_Minutes'] = match_dataset_model['Away_Minutes'].to_list()
fact_match_statistics['Away_Gls'] = match_dataset_model['Away_Gls'].to_list()
fact_match_statistics['Away_Ast'] = match_dataset_model['Away_Ast'].to_list()
fact_match_statistics['Away_PK'] = match_dataset_model['Away_PK'].to_list()
fact_match_statistics['Away_PK_Att'] = match_dataset_model['Away_PK_Att'].to_list()
fact_match_statistics['Away_Sh'] = match_dataset_model['Away_Sh'].to_list()
fact_match_statistics['Away_SoT'] = match_dataset_model['Away_SoT'].to_list()
fact_match_statistics['Away_CrdY'] = match_dataset_model['Away_CrdY'].to_list()
fact_match_statistics['Away_CrdR'] = match_dataset_model['Away_CrdR'].to_list()
fact_match_statistics['Away_Touches'] = match_dataset_model['Away_Touches'].to_list()
fact_match_statistics['Away_Tkl'] = match_dataset_model['Away_Tkl'].to_list()
fact_match_statistics['Away_Int'] = match_dataset_model['Away_Int'].to_list()
fact_match_statistics['Away_Blocks'] = match_dataset_model['Away_Blocks'].to_list()
fact_match_statistics['Away_Touches'] = match_dataset_model['Away_Touches'].to_list()
fact_match_statistics['Away_Tkl'] = match_dataset_model['Away_Tkl'].to_list()
fact_match_statistics['Away_Int'] = match_dataset_model['Away_Int'].to_list()
fact_match_statistics['Away_Blocks'] = match_dataset_model['Away_Blocks'].to_list()
fact_match_statistics['Away_xG'] = match_dataset_model['Away_xG'].to_list()
fact_match_statistics['Away_npxG'] = match_dataset_model['Away_npxG'].to_list()
fact_match_statistics['Away_xAG'] = match_dataset_model['Away_xAG'].to_list()
fact_match_statistics['Away_SCA'] = match_dataset_model['Away_SCA'].to_list()
fact_match_statistics['Away_GCA'] = match_dataset_model['Away_GCA'].to_list()
fact_match_statistics['Away_Cmp_Passes'] = match_dataset_model['Away_Cmp_Passes'].to_list()
fact_match_statistics['Away_Att_Passes'] = match_dataset_model['Away_Att_Passes'].to_list()
fact_match_statistics['Away_Cmp_percent_Passes'] = match_dataset_model['Away_Cmp_percent_Passes'].to_list()
fact_match_statistics['Away_PrgP_Passes'] = match_dataset_model['Away_PrgP_Passes'].to_list()
fact_match_statistics['Away_Carries_Carries'] = match_dataset_model['Away_Carries_Carries'].to_list()
fact_match_statistics['Away_PrgC_Carries'] = match_dataset_model['Away_PrgC_Carries'].to_list()
fact_match_statistics['Away_Att_Take_Ons'] = match_dataset_model['Away_Att_Take_Ons'].to_list()
fact_match_statistics['Away_Succ_Take_Ons'] = match_dataset_model['Away_Succ_Take_Ons'].to_list()

fact_match_statistics['Home_Attack'] = match_dataset_model['Home_Attack'].to_list()
fact_match_statistics['Home_Midfield'] = match_dataset_model['Home_Midfield'].to_list()
fact_match_statistics['Home_Defense'] = match_dataset_model['Home_Defense'].to_list()
fact_match_statistics['Away_Attack'] = match_dataset_model['Away_Attack'].to_list()
fact_match_statistics['Away_Midfield'] = match_dataset_model['Away_Midfield'].to_list()
fact_match_statistics['Away_Defense'] = match_dataset_model['Away_Defense'].to_list()

fact_match_statistics['Home_Score'] = match_dataset_model['Home_Score'].to_list()
fact_match_statistics['Away_Score'] = match_dataset_model['Away_Score'].to_list()

<h3>3. Fact Players Statistics</h3>

In [ ]:
fact_players_statistics = pd.DataFrame({'Match': [],
                                        'Team': [],
                                        'Player_Name': [], 'Player_Kitnum': [],
                                        'Nationality': [], 'Position': [],
                                        'Age': [], 'Minutes': [],
                                        'Gls': [], 'Ast': [], 'PK': [],
                                        'PK_Att': [], 'Sh': [], 'SoT': [],
                                        'CrdY': [], 'CrdR': [], 'Touches': [],
                                        'Tkl': [], 'Int': [], 'Blocks': [],
                                        'xG': [], 'npxG': [], 'xAG': [],
                                        'SCA': [], 'GCA': [], 'Cmp_Passes': [],
                                        'Att_Passes': [], 'Cmp_percent_Passes': [], 'PrgP_Passes': [],
                                        'Carries_Carries': [], 'PrgC_Carries': [], 'Att_Take_Ons': [],
                                        'Succ_Take_Ons': []})

list_match = []
list_team = []

for i in range(0,len(fbref_matchplayerstats)):
    print(i)

    temp_df = dim_match[dim_match['Match_Id'] == fbref_matchplayerstats['Match_Id'].iloc[i]]

    if (len(temp_df) > 0):
        list_match.append(temp_df['Match_Key'].iloc[0])
    else:
        list_match.append(np.nan)

    temp_df = dim_team[dim_team['Team_Name'] == fbref_matchplayerstats['Team'].iloc[i]]
    if (len(temp_df) > 0):
        list_team.append(temp_df['Team_Key'].iloc[0])
    else:
        list_team.append(np.nan)

#continue

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
364519
364520
364521
364522
364523
364524
364525
364526
364527
364528
364529
364530
364531
364532
364533
364534
364535
364536
364537
364538
364539
364540
364541
364542
364543
364544
364545
364546
364547
364548
364549
364550
364551
364552
364553
364554
364555
364556
364557
364558
364559
364560
364561
364562
364563
364564
364565
364566
364567
364568
364569
364570
364571
364572
364573
364574
364575
364576
364577
364578
364579
364580
364581
364582
364583
364584
364585
364586
364587
364588
364589
364590
364591
364592
364593
364594
364595
364596
364597
364598
364599
364600
364601
364602
364603
364604
364605
364606
364607
364608
364609
364610
364611
364612
364613
364614
364615
364616
364617
364618
364619
364620
364621
364622
364623
364624
364625
364626
364627
364628
364629
364630
364631
364632
364633
364634
364635
364636
364637
364638
364639
364640
364641
364642
364643
364644
364645
364646
364647
364648
364649
364650
364651
364652
36465

In [ ]:
fact_players_statistics['Match'] = list_match
fact_players_statistics['Team'] = list_team

fact_players_statistics['Player_Name'] = fbref_matchplayerstats['Player_Name'].to_list()
fact_players_statistics['Player_Kitnum'] = fbref_matchplayerstats['Player_Kitnum'].to_list()
fact_players_statistics['Nationality'] = fbref_matchplayerstats['Nationality'].to_list()
fact_players_statistics['Position'] = fbref_matchplayerstats['Position'].to_list()

fact_players_statistics['Age'] = fbref_matchplayerstats['Age'].to_list()
fact_players_statistics['Minutes'] = fbref_matchplayerstats['Minutes'].to_list()
fact_players_statistics['Gls'] = fbref_matchplayerstats['Gls'].to_list()
fact_players_statistics['Ast'] = fbref_matchplayerstats['Ast'].to_list()
fact_players_statistics['PK'] = fbref_matchplayerstats['PK'].to_list()
fact_players_statistics['PK_Att'] = fbref_matchplayerstats['PK_Att'].to_list()
fact_players_statistics['Sh'] = fbref_matchplayerstats['Sh'].to_list()
fact_players_statistics['SoT'] = fbref_matchplayerstats['SoT'].to_list()
fact_players_statistics['CrdY'] = fbref_matchplayerstats['CrdY'].to_list()
fact_players_statistics['CrdR'] = fbref_matchplayerstats['CrdR'].to_list()
fact_players_statistics['Touches'] = fbref_matchplayerstats['Touches'].to_list()
fact_players_statistics['Tkl'] = fbref_matchplayerstats['Tkl'].to_list()
fact_players_statistics['Int'] = fbref_matchplayerstats['Int'].to_list()
fact_players_statistics['Blocks'] = fbref_matchplayerstats['Blocks'].to_list()
fact_players_statistics['xG'] = fbref_matchplayerstats['xG'].to_list()
fact_players_statistics['npxG'] = fbref_matchplayerstats['npxG'].to_list()
fact_players_statistics['xAG'] = fbref_matchplayerstats['xAG'].to_list()
fact_players_statistics['SCA'] = fbref_matchplayerstats['SCA'].to_list()
fact_players_statistics['GCA'] = fbref_matchplayerstats['GCA'].to_list()
fact_players_statistics['Passes_Cmp'] = fbref_matchplayerstats['Passes_Cmp'].to_list()
fact_players_statistics['Passes_Att'] = fbref_matchplayerstats['Passes_Att'].to_list()
fact_players_statistics['Passes_Cmp_Percentage'] = fbref_matchplayerstats['Passes_Cmp_Percentage'].to_list()
fact_players_statistics['Passes_PrgP'] = fbref_matchplayerstats['Passes_PrgP'].to_list()
fact_players_statistics['Carries'] = fbref_matchplayerstats['Carries'].to_list()
fact_players_statistics['Carries_PrgC'] = fbref_matchplayerstats['Carries_PrgC'].to_list()
fact_players_statistics['Take_Ons_Att'] = fbref_matchplayerstats['Take_Ons_Att'].to_list()
fact_players_statistics['Take_Ons_Succ'] = fbref_matchplayerstats['Take_Ons_Succ'].to_list()

In [ ]:
#dim_match_squad.to_csv('gs://football-data-etl/football-data-staging/dim_match_squad.csv', index=False)'

In [ ]:
dim_league.to_csv('gs://football_data_etl/football_data_staging/dim_league.csv', index=False)
dim_team.to_csv('gs://football_data_etl/football-data-staging/dim_team.csv', index=False)
dim_date.to_csv('gs://football_data_etl/football-data-staging/dim_date.csv', index=False)
dim_match.to_csv('gs://football_data_etl/football-data-staging/dim_match.csv', index=False)
dim_player.to_csv('gs://football_data_etl/football-data-staging/dim_player.csv', index=False)
dim_player_version.to_csv('gs://football_data_etl/football-data-staging/dim_player_version.csv', index=False)
dim_match_squad.to_csv('gs://football_data_etl/football-data-staging/dim_match_squad.csv', index=False)
dim_match_goals.to_csv('gs://football_data_etl/football-data-staging/dim_match_goals.csv', index=False)

fact_player_attributes.to_csv('gs://football_data_etl/football-data-staging/fact_player_attributes.csv', index=False)
fact_match_statistics.to_csv('gs://football_data_etl/football-data-staging/fact_match_statistics.csv', index=False)
fact_players_statistics.to_csv('gs://football_data_etl/football-data-staging/fact_players_statistics.csv', index=False)